# 📗 파인튜닝 데이터 다듬기: 증강·품질 필터·chat template·분할

교안 01에서 만든 노무 상담 Q&A를 학습에 넣을 수 있는 학습·검증 파일로 다듬습니다. 먼저 증강으로 질문 표현을 늘리고 품질 필터로 거른 뒤, 평가 질문과 겹치는 행을 오염 검사로 빼고 데이터셋의 구성을 점검해 균형을 맞춥니다.

남은 행을 모델이 읽는 학습 텍스트(chat 포맷)로 바꿔 chat template 정합과 토큰 길이를 확인한 뒤 학습/검증으로 나눕니다. 다음 시간에는 이 학습·검증 파일로 지도 파인튜닝(supervised fine-tuning, SFT)을 합니다.

<img src="images/10_lesson02_goals_1.png" width="1050" alt="제목은 '이번 교안의 핵심 1: 질문을 늘리고, 거르고, 오염을 빼기'입니다. 판 세 개가 굵은 화살표로 이어집니다. '1. 증강' 판에서는 원본 질문 '1년간 80퍼센트 이상 출근하면 연차가 며칠 생기나요?'가 '패러프레이즈'와 '역번역' 두 갈래로 새 질문 카드가 되고, 두 카드가 '답은 원본 그대로'로 모이며, 맨 아래에 '숫자 1, 80이 같아야 통과'가 있습니다. '2. 품질 필터 네 단계' 판에서는 네 단으로 좁아지는 깔때기에서 단마다 빨강 카드가 빠지며 '형식 검증: 빈 칸', '길이 필터: 짧은 답', '정확 중복: 같은 질문과 답', '유사 중복: 자카드 0.8 이상'이 적혀 있고, 맨 아래로 초록 카드 더미가 나오며 '단계별 제거 수 비교'가 있습니다. '3. 오염 검사' 판에서는 '평가 질문'과 '학습 질문' 카드가 노랑 상자 '겹침 0.86 (기준 0.7 이상)'으로 이어지고, 같은 색 띠 세 칸 '원본', '패러프레이즈', '역번역'으로 된 묶음에 빨강 엑스가 그어지며 '원본 묶음째 빼기'가 적혀 있습니다. 맨 아래 띠에는 '질문 표현을 늘리고, 결함 있는 행을 거른 뒤, 평가와 겹친 묶음을 뺍니다'가 적혀 있습니다.">

<img src="images/10_lesson02_goals_2.png" width="1050" alt="제목은 '이번 교안의 핵심 2: 균형을 맞추고 학습 텍스트로 바꿔 학습/검증으로 나누기'입니다. 판 세 개가 굵은 화살표로 이어집니다. '4. 구성 점검과 균형' 판에서는 높이가 다른 주제 막대 네 개 위로 점선 '주제 상한: 중앙값의 1.5배'가 지나고, 가장 높은 막대는 점선에서 끝나며 그 위의 주황 블록 두 개에 빨강 엑스와 칩 '증강 행부터 빼기'가 있습니다. '5. chat 포맷과 정합' 판에서는 'Q&A 한 행'이 'messages'가 되고, 위 띠 '추론 프롬프트'의 칸 'system', 'user', 'assistant' 아래에 띠 '학습 텍스트'가 같은 세 칸 뒤에 초록 칸 '답'을 하나 더 가집니다. 그 아래 노랑 상자 '학습 텍스트가 추론 프롬프트로 시작'에 초록 체크가 있습니다. '6. 원본 단위 분할과 저장' 판에서는 같은 색 카드 세 장씩인 묶음 'A', 'B', 'C'가 '학습' 상자로, 'D'가 '검증' 상자로 통째로 들어가고, 두 상자 아래에 보라 파일 아이콘이 하나씩 있으며, 초록 배지 '누수 0'과 노랑 상자 '같은 원본의 행은 한쪽에만'이 있습니다. 맨 아래 띠에는 '주제 균형을 맞추고 chat 포맷으로 바꾼 뒤, 원본 묶음째 학습/검증으로 나눕니다'가 적혀 있습니다.">

**학습 목표**

1. **패러프레이즈와 역번역으로 질문 표현을 늘리고, 원본과 숫자가 같은지 확인합니다.**
2. **형식 검증·길이 필터·정확 중복 제거·유사 중복 제거를 차례로 적용하고 단계별 제거 수를 비교합니다.**
3. **평가 질문과 글자가 많이 겹치는 행을 찾아 그 원본 묶음을 오염 검사로 뺍니다.**
4. **방법별·주제별 행 수와 같은 답 반복 수로 데이터셋 구성을 점검하고, 주제별 상한으로 균형을 맞춥니다.**
5. **남은 행을 chat 포맷으로 바꾸고, 학습 텍스트가 추론 프롬프트로 시작하는지(chat template 정합) 토크나이저로 확인한 뒤 토큰 길이를 셉니다.**
6. **원본 Q&A 단위로 학습/검증을 나눠 같은 원본의 행이 양쪽에 함께 들어가는 누수를 막고, 데이터 카드와 함께 저장합니다.**

## 1. 실습 준비

### 1-1. 경로와 생성용 LLM

증강에 쓸 생성용 LLM과 자료 경로를 교안 01과 같은 셀로 준비합니다.

In [ ]:
import json
import os
from pathlib import Path

import pandas as pd
from dotenv import find_dotenv, load_dotenv
from langchain_openai import ChatOpenAI

material_dir = Path(".")  # data가 있는 단원 폴더. 정답 노트북에서는 ".."
data_dir = material_dir / "data"
output_dir = material_dir / "output"  # 이 단원이 만든 데이터셋 파일을 저장하는 폴더
output_dir.mkdir(exist_ok=True)

# API 키 읽기: 단원 폴더 .env를 먼저 읽고, 없는 키만 상위 실습자료 .env에서 채웁니다
load_dotenv(find_dotenv(usecwd=True))
load_dotenv(material_dir.resolve().parent / ".env")

# 데이터를 만드는 생성용 LLM. .env에 OPENAI_CHAT_MODEL이 없으면 기본 모델을 씁니다
# use_responses_api=True: 앞 단원들과 같이 OpenAI Responses API로 호출합니다
model_name = os.getenv("OPENAI_CHAT_MODEL", "gpt-5.6-luna")
llm = ChatOpenAI(model=model_name, use_responses_api=True)

# 테이블의 긴 글(답, 조문)이 50자에서 잘리지 않고 끝까지 보이게 합니다
pd.set_option("display.max_colwidth", None)

print("생성용 LLM:", model_name)

### 1-2. 교안 01 데이터 불러오기

교안 01이 저장한 `output/labor_qa.jsonl`을 다시 만들지 않고 그대로 불러옵니다. JSONL 파일은 한 줄에 JSON 하나이므로 줄마다 `json.loads`로 읽습니다. 따라하기 데이터도 같은 방법으로 읽으므로 `load_jsonl` 함수로 둡니다.

- **`method`:** 교안 01에서 행을 만든 방법입니다. 문서 생성은 조문을 읽고 바로 만든 Q&A, 질문 늘리기는 교안 01에서 늘린 새 질문에 검색으로 찾은 조문을 보고 생성용 LLM이 답을 쓴 행, 범위 밖은 거절 행입니다. 범위 밖 행은 근거 조문이 없어 `doc_id`가 "없음"이고, 답은 고정 문장인 거절 답입니다.
- **`origin_id`:** 행이 나온 원본 Q&A의 id입니다. 원본 행은 자기 id이고, 증강으로 늘린 행은 원본의 id를 물려받습니다. 같은 `origin_id`를 가진 행들(원본과 그 증강 행)을 원본 묶음이라 부릅니다.
- 학습/검증을 나눌 때 같은 원본에서 나온 행을 한쪽에만 넣는 데 이 값을 씁니다.

In [ ]:
def load_jsonl(path):
    """JSONL 파일을 줄마다 읽어 행 딕셔너리(dict) 리스트로 돌려줍니다(빈 줄은 건너뜁니다)."""
    rows = []
    with open(path, encoding="utf-8") as file:
        for line in file:
            if line.strip() != "":
                rows.append(json.loads(line))
    return rows


# 교안 01이 저장한 Q&A 데이터셋을 읽습니다
qa_rows = load_jsonl(output_dir / "labor_qa.jsonl")

# 원본 Q&A는 자기 id가 곧 origin_id입니다
for row in qa_rows:
    row["origin_id"] = row["id"]

qa_table = pd.DataFrame(qa_rows)
print("Q&A 행 수:", len(qa_rows))
display(qa_table.head())

## 2. 증강

사용자는 같은 내용을 여러 말투로 묻습니다. 질문 늘리기는 새 상황의 질문이라 생성용 LLM의 답이 새로 필요했고, 증강은 같은 질문의 표현만 바꿔 원본 답을 그대로 씁니다.

**증강(augmentation)**
- 뜻은 그대로 두고 표현만 바꾼 행을 더해 데이터를 늘리는 일입니다. PyTorch 신경망 단원에서 그림에 적용한 데이터 증강(data augmentation)을 글에 적용합니다.
- 목적은 말투가 달라도 같은 답을 하도록 학습시키는 것입니다.

<img src="images/12_augment_keep_meaning.png" width="1050" alt="제목은 '증강: 질문 표현만 바꾸고 답은 그대로 둡니다'입니다. 왼쪽에 원본 질문 카드 '1년간 80퍼센트 이상 출근하면 연차가 며칠 생기나요?'와 답 카드 '15일의 유급휴가가 생깁니다.'가 있습니다. 세 갈래 가운데 '패러프레이즈'는 '출근율이 80퍼센트 이상이면 1년에 연차를 며칠 받을 수 있나요?'로, '역번역 (한국어 -> 영어 -> 한국어)'은 '1년 동안 80퍼센트 이상 출근한 경우 연차는 며칠인가요?'로 이어지고 둘 다 초록 '통과'입니다. '숫자가 바뀐 질문'은 '90퍼센트'에 빨간 동그라미를 친 '출근율이 90퍼센트 이상이면 연차가 며칠 생기나요?' 카드로 이어지고, 빨강 '걸림'입니다. 아래 상자 두 개에는 '답은 원본 그대로'와 '숫자 1, 80 이 같으면 통과'가 적혀 있습니다.">

연차 질문 하나로 따라가 봅니다.

1. **원본:** "1년간 80퍼센트 이상 출근하면 연차가 며칠 생기나요?"이고 답은 15일입니다.
2. **증강:** 같은 뜻의 "출근율이 80퍼센트 이상이면 1년에 연차를 며칠 받을 수 있나요?"를 만들고, 답은 원본 그대로 붙입니다.
3. **확인:** 새 질문의 숫자(1, 80)가 원본과 같은지 봅니다. "90퍼센트"로 바뀐 질문은 다른 것을 묻는 질문이라 버립니다.

이 교안은 **질문만 바꾸고 답은 원본 그대로** 둡니다. 답을 바꾸다 숫자나 조건이 달라지면 모델이 틀린 사실을 학습하기 때문입니다.

### 2-1. 패러프레이즈

**패러프레이즈(paraphrase):** 같은 뜻을 다른 낱말과 어순으로 바꿔 쓴 문장입니다. 교안 01과 같은 방법(프롬프트 템플릿 `|` 구조화 출력)으로 체인을 만듭니다.

- **`Paraphrase`:** 바꿔 쓴 문장 하나(`text`)입니다. 문장 하나만 받아도 구조화 출력을 쓰면 "바꾼 문장:" 같은 머리말이나 따옴표가 섞이지 않고 문장만 돌아와, 그대로 학습 데이터에 넣을 수 있습니다.
- **`paraphrase_prompt`:** `{text}`(원문)를 채우는 프롬프트 템플릿입니다. 숫자·기간·이름을 그대로 두라는 규칙을 넣지만 지켜지지 않을 수 있어, 뒤에서 숫자를 다시 확인합니다.

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from pydantic import BaseModel, Field


class Paraphrase(BaseModel):
    """같은 뜻을 다른 표현으로 쓴 문장 하나입니다."""
    text: str = Field(description="뜻은 같고 표현(어순, 낱말, 말투)만 다른 문장 하나")


# 숫자, 기간, 이름이 바뀌면 다른 것을 묻는 질문이 되므로 그대로 두라고 합니다
paraphrase_prompt = ChatPromptTemplate.from_messages([
    ("system", "노무 상담 챗봇 학습 데이터의 질문을 뜻은 그대로 두고 다른 표현으로 바꿔 써야 한다. 바꿔 쓴 질문에는 원래 질문의 답이 그대로 붙는다."),
    ("human", """아래 [문장]을 뜻은 그대로 두고 표현(어순, 낱말, 말투)만 바꿔 한 문장으로 다시 써라.

뜻 지키기:
1. 숫자, 기간, 이름은 원문 그대로 둔다.
2. 새로운 정보를 더하거나 빼지 않는다.

말투:
3. 직장인이나 사업주가 실제로 물어볼 말투로 쓴다.

[문장]
{text}"""),
])

# 프롬프트 -> 생성용 LLM(Paraphrase 모양으로만 답함) 체인
paraphrase_generator = paraphrase_prompt | llm.with_structured_output(Paraphrase)

질문마다 표현 하나만 만듭니다. 한 질문에서 여러 개를 받으면 서로 비슷한 문장이 늘어 유사 중복 제거에서 빠지기 쉽습니다.

- 교안 01처럼 `.batch`에 `max_concurrency`와 `return_exceptions=True`를 주어, 실패한 요청은 결과 자리에 예외 객체로 받고 건너뜁니다.
- 새 행은 원본 행을 복사해 `id`·`method`·`question`만 바꿉니다. 답·`origin_id`·주제 태그가 원본 그대로 남습니다.
- **범위 밖 행은 증강하지 않습니다:** 서로 다른 질문 여러 개가 이미 같은 거절 답을 가지므로 표현을 더 늘릴 필요가 없습니다. 근거 조문으로 답한 행(`grounded_rows`)만 늘리고 뒤에서 기존 FAQ도 더하므로, 범위 밖 행의 비율은 교안 01에서 저장할 때보다 낮아집니다. 이 비율은 품질 필터와 오염 검사를 마친 뒤 구성 점검의 방법별 테이블에서 확인합니다.

In [ ]:
# .batch로 동시에 처리할 요청 수의 상한. 교안 01에서 Q&A를 생성할 때와 같은 값이고, 429 오류(요청 한도 초과)가 나면 줄입니다
MAX_CONCURRENCY = 32

# 범위 밖 행은 빼고 근거 조문으로 답한 행만 증강 대상으로 모읍니다
grounded_rows = []
for row in qa_rows:
    if row["method"] != "범위 밖":
        grounded_rows.append(row)

# 질문마다 패러프레이즈 하나를 요청합니다(모델 호출: grounded_rows 행 수만큼)
paraphrase_inputs = [{"text": row["question"]} for row in grounded_rows]
print("증강 대상 행 수:", len(grounded_rows), "| 첫 입력:", paraphrase_inputs[0]["text"])
paraphrase_results = paraphrase_generator.batch(paraphrase_inputs, config={"max_concurrency": MAX_CONCURRENCY}, return_exceptions=True)

paraphrase_rows = []
paraphrase_skipped = 0
for row, result in zip(grounded_rows, paraphrase_results):
    # 실패한 요청은 결과 자리에 예외 객체가 있으므로 건너뜁니다
    if isinstance(result, Exception):
        paraphrase_skipped += 1
        continue

    # 원본 행을 복사해 바뀌는 키만 고칩니다
    new_row = dict(row)
    new_row["id"] = f"{row['id']}-p"
    new_row["method"] = "패러프레이즈"
    new_row["question"] = result.text
    paraphrase_rows.append(new_row)

# 원본 질문을 id로 찾는 딕셔너리(역번역과 뜻 보존 확인에서도 씁니다)
question_by_id = {row["id"]: row["question"] for row in qa_rows}

# 새 질문(question) 옆에 원본 질문을 붙여 표현이 어떻게 바뀌었는지 봅니다
paraphrase_table = pd.DataFrame(paraphrase_rows)
paraphrase_table["원본 질문"] = [question_by_id[row["origin_id"]] for row in paraphrase_rows]
print("패러프레이즈 행 수:", len(paraphrase_rows), "| 건너뛴 요청 수:", paraphrase_skipped)
display(paraphrase_table.head())

### 2-2. 역번역

**역번역(back-translation):** 문장을 다른 언어로 번역했다가 원래 언어로 다시 번역해 표현을 바꾸는 방법입니다. 이름은 기계 번역의 학습 데이터를 늘리던 기법([Sennrich et al., 2016](https://arxiv.org/abs/1511.06709))에서 왔고, 왕복 번역으로 표현을 바꾸는 증강은 그 응용입니다.

- **`Translated`, `translate_prompt`, `translator`:** 번역문 하나(`text`)를 받는 스키마, `{language}`(번역할 언어)·`{text}`(원문)를 채우는 프롬프트 템플릿, 둘을 이은 체인입니다. 같은 체인을 영어 번역과 한국어 번역에 함께 씁니다.
- **표본만:** 질문 하나에 호출이 두 번이라 전체에 하면 패러프레이즈의 두 배가 듭니다. 그래서 역번역 행은 증강 대상(`grounded_rows`)의 표본 200개에서만 만들어 더합니다.

In [ ]:
import random


class Translated(BaseModel):
    """번역 결과입니다."""
    text: str = Field(description="번역된 문장 한 개")


# 숫자가 바뀌지 않게 하고, 설명 없이 번역문만 받습니다
translate_prompt = ChatPromptTemplate.from_messages([
    ("system", "주어진 문장을 자연스럽게 번역해야 한다."),
    ("human", """아래 [문장]을 {language}로 자연스럽게 번역해라. 숫자는 그대로 두고 번역문만 써라.

[문장]
{text}"""),
])

# 프롬프트 -> 생성용 LLM(Translated 모양으로만 답함) 체인
translator = translate_prompt | llm.with_structured_output(Translated)

In [ ]:
# 그림의 연차 질문 하나로 영어 -> 한국어 왕복을 먼저 확인합니다(모델 호출 2회)
sample_question = "1년간 80퍼센트 이상 출근하면 연차가 며칠 생기나요?"
print("원본:", sample_question)
sample_english = translator.invoke({"language": "영어", "text": sample_question}).text
sample_back = translator.invoke({"language": "한국어", "text": sample_english}).text
print("영어:", sample_english)
print("역번역:", sample_back)

표본 200개는 영어 번역을 한꺼번에 받은 뒤, 성공한 영어 문장만 다시 한꺼번에 한국어로 번역합니다. 두 번째 `.batch`는 영어 번역에 성공한 행만 받으므로, 그 행 리스트를 따로 두어 결과와 다시 짝짓습니다.

In [ ]:
# 같은 입력 파일에서 같은 시드면 같은 행이 뽑힙니다
SAMPLE_SEED = 42
back_translation_sample = random.Random(SAMPLE_SEED).sample(grounded_rows, 200)

# 1차: 표본 질문을 영어로 번역합니다(모델 호출 200회)
english_inputs = [{"language": "영어", "text": row["question"]} for row in back_translation_sample]
print("표본 수:", len(back_translation_sample), "| 첫 입력:", english_inputs[0]["text"])
english_results = translator.batch(english_inputs, config={"max_concurrency": MAX_CONCURRENCY}, return_exceptions=True)

# 영어 번역에 성공한 행과 영어 문장만 같은 순서로 모읍니다
english_rows = []
english_texts = []
for row, result in zip(back_translation_sample, english_results):
    if isinstance(result, Exception):
        continue
    english_rows.append(row)
    english_texts.append(result.text)

# 2차: 영어 문장을 다시 한국어로 번역합니다(모델 호출: 영어 번역에 성공한 수만큼)
korean_inputs = [{"language": "한국어", "text": text} for text in english_texts]
korean_results = translator.batch(korean_inputs, config={"max_concurrency": MAX_CONCURRENCY}, return_exceptions=True)

# 패러프레이즈와 같은 방법으로 역번역 행을 만듭니다
back_translation_rows = []
for row, result in zip(english_rows, korean_results):
    if isinstance(result, Exception):
        continue
    new_row = dict(row)
    new_row["id"] = f"{row['id']}-b"
    new_row["method"] = "역번역"
    new_row["question"] = result.text
    back_translation_rows.append(new_row)

# 역번역 질문(question) 옆에 원본 질문을 붙여 봅니다
back_translation_table = pd.DataFrame(back_translation_rows)
back_translation_table["원본 질문"] = [question_by_id[row["origin_id"]] for row in back_translation_rows]
print("영어 번역 성공 수:", len(english_rows), "| 역번역 행 수:", len(back_translation_rows))
display(back_translation_table.head())

### 2-3. 뜻 보존 확인

증강한 질문이 원본과 같은 것을 묻는지 확인한 뒤에 데이터셋에 넣습니다.

- **숫자 같음:** 원본 질문과 증강 질문에서 숫자(`re.findall(r"\d+", ...)`)를 뽑아 정렬한 리스트가 같아야 합니다. 숫자가 빠지거나 바뀐 질문은 다른 것을 묻습니다.
- **원본과 같은 문장:** 표현을 늘리지 못하지만 답도 원본 그대로라, 뒤의 정확 중복 제거가 걸러 냅니다.
- **사람이 볼 것:** "언제"가 "누가"로 바뀌는 것처럼 숫자 없이 뜻이 바뀐 질문은 코드로 잡지 못하므로, 통과한 행도 일부를 원본과 나란히 읽어 봅니다.
- **뜻이 같은데 걸리는 질문:** "두 번"을 "2번"으로, "1년"을 "연간"으로 바꾼 것처럼 숫자를 다르게 적으면 뜻이 같아도 숫자 리스트가 달라 걸립니다. 걸린 행을 읽어 뜻이 같으면 다시 넣을지 판정합니다.

In [ ]:
import re

# 원본 질문은 패러프레이즈에서 만든 question_by_id로 찾습니다
augmented_rows = []
augment_dropped = []
for row in paraphrase_rows + back_translation_rows:
    original_question = question_by_id[row["origin_id"]]

    # 숫자 리스트를 정렬해 비교하면 숫자가 빠지거나 바뀌거나 더해진 경우를 모두 잡습니다
    same_numbers = sorted(re.findall(r"\d+", original_question)) == sorted(re.findall(r"\d+", row["question"]))

    if same_numbers:
        augmented_rows.append(row)
    else:
        augment_dropped.append({"id": row["id"], "원본 질문": original_question, "증강 질문": row["question"]})

augment_dropped_table = pd.DataFrame(augment_dropped, columns=["id", "원본 질문", "증강 질문"])
print("통과한 증강 행 수:", len(augmented_rows), "| 숫자가 달라 걸린 행 수:", len(augment_dropped))
display(augment_dropped_table.head(10))

**결과 읽기**
- **걸린 행:** 원본과 증강 질문의 숫자를 나란히 대 봅니다. 걸린 수는 실행마다 다릅니다.
- **남은 영어 낱말:** 역번역은 "four명"처럼 영어 낱말이 남을 수 있고, 숫자가 없는 질문에 남은 영어는 숫자 확인을 통과하므로 통과한 역번역 행도 몇 개 읽습니다.

## 3. 품질 필터

학습 데이터에 빈 칸·짧은 답·중복이 섞이면 모델이 그 결함까지 학습하므로 학습 전에 행을 거릅니다. 생성한 데이터에 예전에 사람이 쓴 FAQ를 합치면 이런 결함이 함께 들어옵니다.

**네 단계와 순서**
- **형식 검증:** 필수 키(`question`·`answer`)가 없거나 값이 비어 있는 행을 버립니다.
- **길이 필터:** 답이 너무 짧은 행을 버립니다.
- **정확 중복 제거:** 질문과 답이 글자까지 같은 행(정확 중복, exact duplicate)을 하나만 남깁니다.
- **유사 중복 제거:** 질문이 거의 같은 행(유사 중복, near-duplicate)을 하나만 남깁니다.

중복 제거는 먼저 나온 행을 남깁니다. 문서 생성·질문 늘리기 행은 근거 조문(`doc_id`)이 있고 교안 01에서 숫자 대조를 거쳤으므로, 겹칠 때 생성 행을 남기려고 원본 Q&A, 증강 행 뒤 맨 끝에 기존 FAQ를 둡니다.

### 3-1. 기존 FAQ 합치기

노무 상담 창구가 예전에 쓰던 기존 FAQ 322건을 합칩니다(수업용으로 만든 자료). 사람이 자주 받은 질문이 담겨 있지만, 오래 관리되며 결함이 섞였습니다. 실제 상담 기록에서 가져온 FAQ라면 합치기 전에 이름, 연락처 같은 개인정보부터 지웁니다.

- id는 `faq-001`부터 붙이고, 원본이 따로 없으므로 `origin_id`는 자기 id입니다.
- 근거 조문 id(`doc_id`)가 없어 테이블에서 빈 칸(`NaN`)으로 나옵니다. `answer` 키가 빠진 행도 그대로 옮겨 형식 검증에서 거르는 모습을 봅니다.
- **`dropped_rows(before, after)`:** 한 단계 전후의 행 리스트를 받아 그 단계에서 버린 행을 돌려줍니다. 뒤에서 만드는 중복 제거 함수처럼 남긴 행만 돌려주는 함수에서도 버린 행을 보려고 품질 필터 네 단계와, 뒤에서 주제별 행 수를 상한으로 줄일 때 함께 씁니다.

In [ ]:
# 기존 FAQ 322건 읽기
faq_old = json.loads((data_dir / "본실습" / "labor_faq_old.json").read_text(encoding="utf-8"))

faq_rows = []
for number, faq in enumerate(faq_old, start=1):
    faq_id = f"faq-{number:03d}"
    row = {"id": faq_id, "origin_id": faq_id, "source": faq["source"], "method": "기존 FAQ"}

    # 키가 빠진 행도 있는 키만 옮겨 그대로 둡니다
    for key in ["question", "answer"]:
        if key in faq:
            row[key] = faq[key]
    faq_rows.append(row)

# 겹칠 때 생성 행을 남기려고 원본 Q&A, 증강 행, 기존 FAQ 순서로 합칩니다
raw = qa_rows + augmented_rows + faq_rows

# 키가 없는 칸은 빈 칸(NaN)으로 나옵니다
raw_table = pd.DataFrame(raw)
print("합친 행 수:", len(raw))
display(raw_table["method"].value_counts())
display(raw_table.tail())

In [ ]:
def dropped_rows(before, after):
    """before 가운데 after에 없는 행(그 단계에서 버린 행)을 돌려줍니다."""
    # 남긴 id를 set으로 모읍니다
    kept_ids = {row["id"] for row in after}

    # before에서 그 id에 없는 행만 모읍니다
    dropped = []
    for row in before:
        if row["id"] not in kept_ids:
            dropped.append(row)
    return dropped

### 3-2. 형식 검증

형식 검증을 맨 앞에 둡니다. `answer` 키가 없는 행이 남아 있으면 뒤 단계에서 `row["answer"]`를 읽다가 `KeyError`로 멈추기 때문입니다.

- **`format_ok(row)`:** 두 키가 모두 있고 값이 공백만 있는 문자열이 아니면 `True`입니다.

<img src="images/13_filter_1_format.png" width="1050" alt="제목은 '품질 필터 1단계: 형식 검증'입니다. 윗줄 왼쪽의 주황 상자 '기존 FAQ 322행'에서 화살표로 단계 카드 네 장 '1 형식 검증: 질문과 답이 비어 있지 않음', '2 길이 필터: 답 5글자 이상', '3 정확 중복 제거: 같은 질문과 답은 처음 하나만', '4 유사 중복 제거: 글자 집합 자카드 0.8 미만만'이 이어지고, 1번 카드만 빨간 테두리로 강조되어 있습니다. 그 아래 분홍 패널 '걸린 행'에는 질문 '회사가 취업규칙을 직원들에게 불리하게 바꾸려면 무엇이 …'와 빨간 점선으로 비어 있는 답 칸, 빨강 배지 '걸림'이 있고, 아래에 '걸린 이유: 답 칸 없음'이 적혀 있습니다. 맨 아래 노랑 띠에는 '남은 313행, 제거 9행 (322 - 9 = 313)'이 적혀 있습니다.">

그림 속 행 수는 기존 FAQ 322행만 거른 수입니다. 합친 데이터에서는 생성 행도 함께 걸립니다.

In [ ]:
def format_ok(row):
    """question, answer 키가 모두 있고 두 값이 공백만 있는 문자열이 아니면 True를 돌려줍니다."""
    for key in ["question", "answer"]:
        if key not in row:
            return False
        if row[key].strip() == "":  # 공백만 있는 값도 빈 값으로 봄
            return False
    return True


# 형식 검증을 통과한 행만 담습니다
after_format = []
for row in raw:
    if format_ok(row):
        after_format.append(row)

format_dropped_table = pd.DataFrame(dropped_rows(raw, after_format))
display(format_dropped_table.head())

기존 FAQ에서 `answer` 키가 없는 5건과 질문이 빈 문자열인 4건, 모두 9건이 걸립니다. 생성한 행도 질문이나 답이 비어 있으면 함께 걸립니다.

### 3-3. 길이 필터

"네."처럼 짧은 답에는 조문 근거나 절차가 없어 모델이 배울 내용이 없습니다.

- **`length_ok(row, min_a=5)`:** 답만 검사해 글자 수가 `min_a`(기본 5) 이상이면 `True`입니다.
- 이 교안은 질문 길이를 거르지 않습니다(데이터에 따라 `len(row["question"])` 조건을 같은 방식으로 더합니다).

<img src="images/13_filter_2_length.png" width="1050" alt="제목은 '품질 필터 2단계: 길이 필터'입니다. 윗줄 왼쪽의 주황 상자 '기존 FAQ 322행'에서 화살표로 단계 카드 네 장 '1 형식 검증: 질문과 답이 비어 있지 않음', '2 길이 필터: 답 5글자 이상', '3 정확 중복 제거: 같은 질문과 답은 처음 하나만', '4 유사 중복 제거: 글자 집합 자카드 0.8 미만만'이 이어집니다. 1번 카드에는 초록 체크가 붙어 있고 2번 카드가 빨간 테두리로 강조되어 있습니다. 그 아래 분홍 패널 '걸린 행'에는 질문 '퇴직금 지급 기한이 따로 정해져 있나요?'와 빨간 동그라미를 친 답 '아니요.', 빨강 배지 '걸림'이 있고, 아래에 '걸린 이유: 답이 5글자 미만'이 적혀 있습니다. 맨 아래 노랑 띠에는 '남은 307행, 제거 6행 (313 - 6 = 307)'이 적혀 있습니다.">

In [ ]:
def length_ok(row, min_a=5):
    """답의 글자 수가 min_a 이상이면 True를 돌려줍니다."""
    return len(row["answer"]) >= min_a


# 길이 필터를 통과한 행만 담습니다
after_len = []
for row in after_format:
    if length_ok(row):
        after_len.append(row)

length_dropped_table = pd.DataFrame(dropped_rows(after_format, after_len))
display(length_dropped_table.head())

기존 FAQ의 짧은 답 6건이 걸립니다.

### 3-4. 정확 중복 제거

같은 (질문, 답) 쌍이 여러 번 들어 있으면 모델이 그 행을 여러 번 학습해 한쪽으로 치우칩니다.

- **`dedup_exact(rows)`:** (질문, 답)을 처음 본 행만 순서대로 남깁니다.

<img src="images/13_filter_3_exact.png" width="1050" alt="제목은 '품질 필터 3단계: 정확 중복 제거'입니다. 윗줄 왼쪽의 주황 상자 '기존 FAQ 322행'에서 화살표로 단계 카드 네 장 '1 형식 검증: 질문과 답이 비어 있지 않음', '2 길이 필터: 답 5글자 이상', '3 정확 중복 제거: 같은 질문과 답은 처음 하나만', '4 유사 중복 제거: 글자 집합 자카드 0.8 미만만'이 이어집니다. 1, 2번 카드에는 초록 체크가 붙어 있고 3번 카드가 빨간 테두리로 강조되어 있습니다. 그 아래 분홍 패널 '걸린 행'에는 같은 문장 '탄력근로제 때문에 월급이 줄어들까 봐 걱정이에요.'가 두 줄 있고, 첫 줄에는 초록 배지 '통과', 빨간 테두리를 친 둘째 줄에는 빨강 배지 '걸림'이 붙어 있으며, 아래에 '걸린 이유: 같은 질문과 답'이 적혀 있습니다. 맨 아래 노랑 띠에는 '남은 299행, 제거 8행 (307 - 8 = 299)'가 적혀 있습니다.">

In [ ]:
def dedup_exact(rows):
    """(질문, 답)이 처음 나온 행만 순서대로 남긴 리스트를 돌려줍니다."""
    seen = set()
    kept = []
    for row in rows:
        # (질문, 답)을 set에 기록해 두면 이미 본 쌍인지 바로 알 수 있습니다
        key = (row["question"], row["answer"])
        if key not in seen:
            seen.add(key)
            kept.append(row)
    return kept


# 길이 필터를 통과한 행에서 정확 중복을 뺍니다
after_dedup = dedup_exact(after_len)

dedup_dropped_table = pd.DataFrame(dropped_rows(after_len, after_dedup))
display(dedup_dropped_table.head())

**결과 읽기**
- **기존 FAQ:** FAQ 안에 두 번씩 들어 있던 8건이 빠집니다.
- **증강 행:** 원본과 같은 문장으로 돌아온 패러프레이즈나 역번역은 답도 같아 여기서 빠집니다.

### 3-5. 유사 중복 제거

"연차휴가는 며칠인가요?"와 "연차 휴가는 며칠 인가요?"는 띄어쓰기만 다른 같은 질문입니다. 이런 유사 중복은 정확 중복 제거로는 걸리지 않습니다.

- **`is_near_dup(a, b, threshold=0.8)`:** `jaccard`로 계산한 유사도가 기준 이상이면 유사 중복입니다.
- **`dedup_similar(rows, threshold=0.8)`:** 이미 남긴 질문과 유사 중복인 행을 버리고 먼저 나온 행만 남깁니다.

<img src="images/13_filter_4_similar.png" width="1050" alt="제목은 '품질 필터 4단계: 유사 중복 제거'입니다. 윗줄 왼쪽의 주황 상자 '기존 FAQ 322행'에서 화살표로 단계 카드 네 장 '1 형식 검증: 질문과 답이 비어 있지 않음', '2 길이 필터: 답 5글자 이상', '3 정확 중복 제거: 같은 질문과 답은 처음 하나만', '4 유사 중복 제거: 글자 집합 자카드 0.8 미만만'이 이어집니다. 1~3번 카드에는 초록 체크가 붙어 있고 4번 카드가 빨간 테두리로 강조되어 있습니다. 그 아래 분홍 패널 '걸린 행'에는 초록 배지 '통과'가 붙은 '성희롱 신고 후 조사 과정에서 2차 피해가 걱정돼요.'와, '성희롱신고'에 빨간 동그라미를 치고 빨강 배지 '걸림'이 붙은 '성희롱신고 후 조사 과정에서 2차 피해가 걱정돼요.'가 있고, 아래에 '걸린 이유: 띄어쓰기만 다름 (자카드 1.0)'이 적혀 있습니다. 맨 아래 노랑 띠에는 '남은 291행, 제거 8행 (299 - 8 = 291)'이 적혀 있습니다.">

In [ ]:
# 교안 01에서 비슷한 지시를 거를 때 쓴 jaccard와 같은 함수입니다
def jaccard(a, b):
    """공백을 뺀 두 문장의 글자 집합으로 계산한 자카드 유사도를 돌려줍니다."""
    chars_a = set(a.replace(" ", ""))
    chars_b = set(b.replace(" ", ""))
    return len(chars_a & chars_b) / len(chars_a | chars_b)


def is_near_dup(a, b, threshold=0.8):
    """두 문장의 자카드 유사도가 threshold 이상이면 True를 돌려줍니다."""
    return jaccard(a, b) >= threshold


def dedup_similar(rows, threshold=0.8):
    """이미 남긴 질문과 유사 중복인 행을 버리고, 먼저 나온 행만 남긴 리스트를 돌려줍니다."""
    kept = []
    for row in rows:
        # 이미 남긴 질문 가운데 하나라도 비슷하면 이 행을 버립니다
        duplicated = False
        for kept_row in kept:
            if is_near_dup(row["question"], kept_row["question"], threshold):
                duplicated = True
                break
        if not duplicated:
            kept.append(row)
    return kept

In [ ]:
# 정확 중복을 뺀 행에서 유사 중복을 뺍니다
clean = dedup_similar(after_dedup)

similar_dropped_table = pd.DataFrame(dropped_rows(after_dedup, clean))
display(similar_dropped_table["method"].value_counts())
display(similar_dropped_table.head())

**결과 읽기**
- **버린 행 읽기:** 글자 집합은 뜻을 보지 않아 적용 조문이 다른 질문도 묶입니다(예: 본인 유산휴가와 배우자 유산휴가처럼 낱말 하나만 다른 질문). 버린 행을 읽어 다른 질문이 잘못 빠지지 않았는지 확인합니다.
- **실무:** 행이 아주 많으면 MinHash와 LSH로 비슷한 쌍만 골라 비교하고([Lee et al., 2022](https://arxiv.org/abs/2107.06499), [Hugging Face 중복 제거 글](https://huggingface.co/blog/dedup)), 뜻이 같은 다른 표현은 문장 임베딩으로 찾습니다.

### 3-6. 단계별 제거 수

단계마다 몇 행이 빠졌는지 한 테이블로 보면 어느 결함이 많은 자료인지 알 수 있습니다.

In [ ]:
# 단계마다 남은 행 수와 그 단계에서 뺀 행 수
filter_counts = pd.DataFrame([
    {"단계": "합친 데이터", "남은 수": len(raw), "제거 수": 0},
    {"단계": "형식 검증", "남은 수": len(after_format), "제거 수": len(raw) - len(after_format)},
    {"단계": "길이 필터", "남은 수": len(after_len), "제거 수": len(after_format) - len(after_len)},
    {"단계": "정확 중복 제거", "남은 수": len(after_dedup), "제거 수": len(after_len) - len(after_dedup)},
    {"단계": "유사 중복 제거", "남은 수": len(clean), "제거 수": len(after_dedup) - len(clean)},
])
display(filter_counts)

**결과 읽기**
- **형식 검증, 길이 필터, 정확 중복 제거:** 걸린 행은 대부분 기존 FAQ 결함(9, 6, 8건)이고, 정확 중복에는 원본과 같은 문장으로 돌아온 증강 행이 더해집니다.
- **유사 중복 제거:** 버린 행을 읽어 다른 질문이 잘못 묶이지 않았는지 확인합니다.

## 4. 오염 검사

**오염(contamination):** 평가에 쓸 질문과 같거나 거의 같은 질문이 학습 데이터에 들어 있는 상태입니다. 모델이 평가 질문을 미리 학습하면 평가 점수가 실제 실력보다 높게 나옵니다.

- **평가 질문:** 이 데이터셋으로 학습한 모델을 평가할 때 쓸 질문 60개(`data/본실습/labor_eval_questions.json`)입니다. 학습·검증 어느 쪽에도 없어야 합니다.
- **검사 순서:** 품질 필터 바로 뒤, 구성 점검과 분할 앞에서 검사합니다. 평가 질문과 겹친 묶음을 먼저 빼야 주제 상한과 분할 단계가 그 묶음을 세지 않습니다. 주제 상한 뒤에 검사하면 상한에서 이미 빠진 오염 묶음은 세지 못해, 데이터 카드에 적는 뺀 묶음 수가 실제보다 작아집니다.

<img src="images/15_decontamination.png" width="1050" alt="제목은 '오염 검사: 평가 질문과 겹치는 원본 묶음을 뺍니다'입니다. 왼쪽 위에 '평가 질문' 카드 '육아휴직을 다 안 썼으면 단축근무를 더 길게 쓸 수 있나요?'가 있고, 왼쪽 아래에 '학습 데이터 질문' 카드 '육아휴직을 다 안 썼으면 단축근무를 더 길게 쓸 수 있는지 알려 주세요.'가 있습니다. 두 카드는 글자 조각 칩 '육아휴', '아휴직', '휴직을'로 이어집니다. 계산 결과 상자에는 '겹침 0.86', '기준 0.7 이상'과 빨강 배지 '오염'이 있습니다. 오른쪽에서는 '원본', '패러프레이즈', '역번역' 카드 묶음이 빨강 엑스로 지워지며 '원본 묶음째 제거'가 적혀 있고, 그 옆에 초록 '최종 학습 데이터' 더미가 있습니다. 서류철을 든 노무 담당자가 계산 결과 상자를 가리킵니다.">

**겹침 계산**
- **글자 3-gram:** 공백을 뺀 문장에서 연속한 세 글자 묶음입니다. "연차휴가"의 3-gram은 "연차휴"와 "차휴가"입니다.
- **`ngram_overlap(question, eval_question)`:** 평가 질문의 3-gram 가운데 학습 질문에도 있는 비율입니다. 평가 질문을 분모로 두어, 평가 질문이 학습 질문 안에 얼마나 들어 있는지 계산합니다.
- **기준:** 행마다 평가 질문 60개와의 겹침 가운데 가장 큰 값이 `CONTAMINATION_THRESHOLD`(0.7) 이상이면 오염으로 봅니다.

[Tulu 3](https://arxiv.org/abs/2411.15124) 같은 공개 학습 레시피도 학습 데이터와 평가 데이터의 n-gram 겹침을 검사해 겹치는 데이터를 뺍니다.

기존 FAQ에는 평가 질문의 끝맺음만 바꿔 옮긴 질문이 섞여 있습니다. 그 가운데 하나와 평가 질문의 겹침을 먼저 계산합니다.

In [ ]:
def char_ngrams(text, n=3):
    """공백을 뺀 문장의 글자 n-gram 집합을 돌려줍니다."""
    compact = text.replace(" ", "")
    ngrams = set()
    for index in range(len(compact) - n + 1):  # n글자씩 한 칸씩 밀며 자름
        ngrams.add(compact[index:index + n])
    return ngrams


def ngram_overlap(question, eval_question):
    """평가 질문의 글자 3-gram 가운데 question에도 있는 비율을 돌려줍니다."""
    eval_ngrams = char_ngrams(eval_question)
    return len(char_ngrams(question) & eval_ngrams) / len(eval_ngrams)  # 평가 질문 기준이라 0~1 사이


# 평가 질문의 끝맺음만 바꾼 기존 FAQ 질문과 그 평가 질문
example_question = "육아휴직을 다 안 썼으면 단축근무를 더 길게 쓸 수 있는지 알려 주세요."
example_eval = "육아휴직을 다 안 썼으면 단축근무를 더 길게 쓸 수 있나요?"
print("평가 질문의 3-gram 수:", len(char_ngrams(example_eval)))
print("FAQ 질문에도 있는 3-gram 수:", len(char_ngrams(example_question) & char_ngrams(example_eval)))
print("겹침:", round(ngram_overlap(example_question, example_eval), 2))

평가 질문의 3-gram 22개 가운데 19개가 FAQ 질문에도 있어 겹침이 0.86입니다. 기준 0.7을 넘으므로 오염입니다. 끝맺음만 바꾸면 평가 질문의 3-gram 대부분이 그대로 남습니다.

이제 품질 필터를 거친 행(`clean`)마다 평가 질문 60개와의 겹침을 모두 계산하고, 가장 큰 값과 그 평가 질문을 함께 기록해 기준 이상인 행을 모읍니다.

In [ ]:
# 평가 질문 60개와 오염 기준
eval_questions = json.loads((data_dir / "본실습" / "labor_eval_questions.json").read_text(encoding="utf-8"))
CONTAMINATION_THRESHOLD = 0.7  # 평가 질문 3-gram의 70% 이상이면 오염

contaminated = []
for row in clean:
    # 평가 질문 60개 가운데 가장 많이 겹치는 질문과 그 겹침
    best_overlap = 0
    best_eval = ""
    for eval_question in eval_questions:
        overlap = ngram_overlap(row["question"], eval_question)
        if overlap > best_overlap:
            best_overlap = overlap
            best_eval = eval_question

    if best_overlap >= CONTAMINATION_THRESHOLD:
        contaminated.append({"id": row["id"], "origin_id": row["origin_id"], "method": row["method"], "question": row["question"], "평가 질문": best_eval, "겹침": best_overlap})

contaminated_table = pd.DataFrame(contaminated, columns=["id", "origin_id", "method", "question", "평가 질문", "겹침"])
print("평가 질문 수:", len(eval_questions), "| 걸린 행 수:", len(contaminated))
display(contaminated_table.round(2))

걸린 행만 빼지 않고 그 행의 **원본 묶음 전체**를 뺍니다. 같은 묶음의 패러프레이즈·역번역도 평가 질문의 다른 표현이기 때문입니다. 뺀 결과(`decontaminated`)로 구성 점검, 균형, 분할을 이어 갑니다.

In [ ]:
# 걸린 행의 원본 묶음
contaminated_origins = {row["origin_id"] for row in contaminated}

# 걸린 원본 묶음을 뺀 행만 담습니다
decontaminated = []
for row in clean:
    if row["origin_id"] not in contaminated_origins:  # 같은 원본의 증강 행까지 함께 뺌
        decontaminated.append(row)

print("뺀 원본 묶음 수:", len(contaminated_origins), "| 오염 검사 전:", len(clean), "-> 후:", len(decontaminated))

뜻이 같아도 표현이 다른 질문은 글자 겹침이 낮아 이 검사로 걸리지 않습니다. 그래서 남은 데이터를 평가 질문 리스트와 사람이 한 번 대조합니다.

## 5. 데이터셋 구성 점검과 균형

품질 필터는 행 하나하나의 결함을 거르지만, 학습 데이터 전체의 구성은 보지 않습니다. 어떤 주제, 어떤 방법의 행이 많은지, 같은 답이 몇 번 나오는지가 한쪽으로 치우치면 모델이 많은 쪽에 맞춰집니다. 실무에서는 학습 전에 구성 테이블을 보고 많은 쪽을 줄이거나 적은 쪽을 채웁니다.

### 5-1. 구성 점검

오염 검사를 마친 행(`decontaminated`)을 세 가지 기준으로 셉니다.

- **방법별:** 문서 생성, 질문 늘리기, 패러프레이즈, 역번역, 기존 FAQ, 범위 밖 행의 수와 비율입니다. 범위 밖 행의 비율도 여기서 봅니다.
- **주제별:** 주제 태그(`source`)별 행 수, 문단 수, 문단당 행 수입니다. 문단 수는 근거 조문(`doc_id`)의 고유 수이고, `doc_id`가 없는 기존 FAQ 행은 문단 수와 문단당 행 수에서 빠집니다. 범위 밖 행은 주제가 없어 뺍니다.
- **같은 답 반복:** 답 문장별 행 수와 원본 묶음의 크기입니다. 같은 원본 묶음(`origin_id`)의 행은 답이 같고, 범위 밖 행은 모두 같은 거절 답을 가집니다.

In [ ]:
# 오염 검사를 마친 행을 테이블로 만들어 방법별 행 수와 비율(%)을 봅니다
decontaminated_table = pd.DataFrame(decontaminated)
method_counts = decontaminated_table["method"].value_counts()
method_table = pd.DataFrame({"행 수": method_counts, "비율(%)": (method_counts / len(decontaminated) * 100).round(1)})  # 비율 = 방법별 행 수 / 전체 행 수 x 100
display(method_table)

In [ ]:
# 범위 밖 행은 주제가 없어 빼고, 주제마다 묶습니다
topic_rows_table = decontaminated_table[decontaminated_table["method"] != "범위 밖"]
topic_groups = topic_rows_table.groupby("source")

# 문단 수는 doc_id의 고유 수입니다. 기존 FAQ 행은 doc_id가 빈 칸(NaN)이라 nunique와 count가 세지 않습니다
topic_table = pd.DataFrame({"행 수": topic_groups.size(), "문단 수": topic_groups["doc_id"].nunique()})
topic_table["문단당 행 수"] = (topic_groups["doc_id"].count() / topic_table["문단 수"]).round(1)
topic_table = topic_table.sort_values("행 수", ascending=False)

# 주제별 행 수는 뒤의 주제 상한에서도 씁니다
topic_counts = topic_table["행 수"]
display(topic_table)

In [ ]:
# 같은 답 문장이 몇 번 나오는지 셉니다(많은 순서로 다섯 개)
display(decontaminated_table["answer"].value_counts().head())

# 원본 묶음마다 행 수(묶음 크기)를 센 뒤, 묶음 크기별로 묶음이 몇 개인지 셉니다
bundle_sizes = decontaminated_table["origin_id"].value_counts()
display(bundle_sizes.value_counts().sort_index().rename_axis("묶음 크기").rename("묶음 수"))

**결과 읽기**
- **방법별과 범위 밖 비율:** 증강 행이 원본보다 많으면 같은 답이 반복되는 행이 많아집니다. 범위 밖 비율의 목표는 최종 데이터의 3~5%이고, 데이터 카드의 최종 비율이 낮으면 교안 01의 `OUT_OF_SCOPE_COUNT`(70)를 늘리고 높으면 줄입니다.
- **주제별:** 문단당 행 수가 다른 주제보다 지나치게 큰 주제는 같은 조문에 질문이 몰린 것이므로, 교안 01의 `MAX_NEW_PER_TOPIC`(30)을 낮춥니다.
- **같은 답 반복:** 한 원본 묶음은 같은 답을 최대 3번 가지고, 거절 답은 범위 밖 행 수만큼 나옵니다. 거절 답 말고 4번 이상 나오는 답은 다른 묶음이 같은 답을 쓴 것이라 그 행들을 읽어 봅니다.

### 5-2. 주제 상한으로 균형 맞추기

행이 많은 주제를 줄여 주제 분포를 고르게 합니다. 적은 주제를 같은 행 복제로 늘리면 같은 답의 반복이 늘어 과적합되므로, 적은 주제는 교안 01의 질문 늘리기로 채웠고 여기서는 많은 주제만 줄입니다.

- **`cap_by_topic(rows, cap, seed)`:** 상한을 넘는 주제는 그 주제의 증강 행(패러프레이즈, 역번역)을 시드로 섞은 순서로 하나씩 빼다가 상한과 같아지면 멈춥니다. 증강 행을 다 빼도 넘으면 원본 묶음을 하나씩 뺍니다. 이때 그 주제의 묶음에는 증강 행이 남아 있지 않아 원본 한 행씩 빠집니다.
- **범위 밖 행과 순서:** 범위 밖 행은 상한 대상이 아니라 모두 남기고, 남긴 행은 원래 순서대로 돌려줍니다.
- **증강 행부터 빼는 이유:** 증강 행은 남아 있는 원본과 같은 답을 반복할 뿐이지만, 원본 Q&A를 빼면 그 질문과 답의 내용이 데이터에서 사라집니다.

In [ ]:
def cap_by_topic(rows, cap, seed):
    """주제별 행 수가 cap을 넘으면 증강 행부터, 그래도 넘으면 원본 묶음째 줄인 행 리스트를 원래 순서대로 돌려줍니다(범위 밖 행은 모두 남김)."""
    # 주제별 행 수를 셉니다. 범위 밖 행은 상한 대상이 아니라 세지 않습니다
    topic_total = {}
    for row in rows:
        if row["source"] != "범위 밖":
            topic_total[row["source"]] = topic_total.get(row["source"], 0) + 1

    # 1단계: 증강 행의 id를 정렬한 뒤 난수 시드로 섞습니다
    topic_by_id = {}
    augmented_ids = []
    for row in rows:
        topic_by_id[row["id"]] = row["source"]
        if row["method"] == "패러프레이즈" or row["method"] == "역번역":
            augmented_ids.append(row["id"])
    augmented_ids = sorted(augmented_ids)
    random.Random(seed).shuffle(augmented_ids)

    # 섞은 순서대로 보며, 그 주제가 아직 cap을 넘으면 증강 행을 하나씩 뺍니다
    removed_ids = set()
    for row_id in augmented_ids:
        topic = topic_by_id[row_id]
        if topic_total[topic] > cap:
            removed_ids.add(row_id)
            topic_total[topic] = topic_total[topic] - 1

    # 2단계: 남은 행으로 원본 묶음마다 주제와 행 수를 셉니다
    bundle_topic = {}
    bundle_size = {}
    for row in rows:
        if row["source"] != "범위 밖" and row["id"] not in removed_ids:
            bundle_topic[row["origin_id"]] = row["source"]
            bundle_size[row["origin_id"]] = bundle_size.get(row["origin_id"], 0) + 1

    # 묶음 이름을 정렬해 섞은 순서대로 보며, 그래도 cap을 넘는 주제는 원본 묶음째 뺍니다
    origins = sorted(bundle_size)
    random.Random(seed).shuffle(origins)
    removed_origins = set()
    for origin in origins:
        topic = bundle_topic[origin]
        if topic_total[topic] > cap:
            removed_origins.add(origin)
            topic_total[topic] = topic_total[topic] - bundle_size[origin]

    # 빼지 않은 행을 원래 순서대로 모읍니다
    kept = []
    for row in rows:
        if row["id"] not in removed_ids and row["origin_id"] not in removed_origins:
            kept.append(row)
    return kept

상한과 시드를 정해 오염 검사를 마친 행(`decontaminated`)에 적용합니다.

- **상한 `TOPIC_CAP`:** 범위 밖을 뺀 주제별 행 수 중앙값의 1.5배를 `round`로 정수로 만든 값(.5는 짝수 쪽)입니다. 1.5배는 이 교안이 정한 값이고, 데이터에 따라 조정합니다.
- **시드 `SPLIT_SEED`(42):** 뒤의 분할에도 같은 값을 쓰고, 따라하기에서도 같은 함수와 시드를 씁니다.

In [ ]:
# 난수 시드. 주제 상한, 원본 단위 분할, 따라하기에 같은 값을 씁니다
SPLIT_SEED = 42

# 상한: 범위 밖을 뺀 주제별 행 수 중앙값의 1.5배
TOPIC_CAP = round(topic_counts.median() * 1.5)  # round는 .5를 짝수 쪽으로 맞춤
balanced = cap_by_topic(decontaminated, TOPIC_CAP, SPLIT_SEED)
print("주제 상한(행):", TOPIC_CAP, "| 균형 전:", len(decontaminated), "-> 균형 후:", len(balanced))

In [ ]:
# 균형에서 뺀 행이 어떤 방법의 행인지 셉니다(뺀 행이 없어도 멈추지 않게 열 이름을 줍니다)
balance_dropped_table = pd.DataFrame(dropped_rows(decontaminated, balanced), columns=["id", "method", "source"])
display(balance_dropped_table["method"].value_counts())

**결과 읽기**
- **뺀 행의 방법:** 상한을 넘은 주제에서는 증강 행이 먼저 빠집니다. 원본 행(문서 생성, 질문 늘리기, 기존 FAQ)이 보이면 증강 행을 모두 빼고도 상한을 넘은 주제입니다.

In [ ]:
# 주제별 행 수를 균형 전후로 나란히 봅니다
balanced_topic_counts = pd.DataFrame(balanced)["source"].value_counts().drop(index="범위 밖", errors="ignore")
balance_table = pd.DataFrame({"균형 전": topic_counts, "균형 후": balanced_topic_counts}).rename_axis("주제").reset_index()
display(balance_table)

**결과 읽기**
- **균형 후:** 상한을 넘던 주제만 상한값으로 줄고, 전체 행이 줄어 범위 밖 비율은 조금 오릅니다(3~5% 목표는 데이터 카드의 최종 비율로 비교).
- 뒤의 chat 포맷, 분할, 저장은 모두 `balanced`로 합니다.

## 6. chat 포맷과 chat template 정합

균형을 맞춘 행(`balanced`)을 모델이 읽는 학습 텍스트로 바꿉니다. Q&A 한 행은 chat 포맷과 chat template을 거쳐 문자열 하나가 되어 모델에 들어갑니다. 첫 행으로 그 문자열을 먼저 확인한 뒤, 전체 행을 바꾸고 토큰 수를 셉니다.

### 6-1. chat 포맷으로 바꾸기

교안 01에서 본 chat 포맷(`messages`, system·user·assistant)으로 바꿉니다. system 문장은 상담 챗봇의 역할을 정하고, 학습과 추론에 같은 문장을 넣어야 하므로 `SYSTEM_PROMPT` 변수 하나에 둡니다.

- **`to_chat(row, system_prompt=SYSTEM_PROMPT)`:** 한 행을 system·user·assistant 세 메시지를 담은 `{"messages": [...]}`로 돌려줍니다. 따라하기에서 다른 system 문장을 넘기려고 인자로 둡니다.

In [ ]:
# 학습과 추론에 똑같이 넣을 system 문장
SYSTEM_PROMPT = "너는 노무 상담 챗봇이다. 노동법 조문에 근거해 존댓말로 간결히 답한다."


def to_chat(row, system_prompt=SYSTEM_PROMPT):
    """Q&A 한 행을 system, user, assistant 세 메시지의 chat 포맷 딕셔너리로 돌려줍니다."""
    return {"messages": [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": row["question"]},
        {"role": "assistant", "content": row["answer"]},
    ]}


# 균형을 맞춘 첫 행을 chat 포맷으로 바꿉니다
sample_chat = to_chat(balanced[0])
display(sample_chat)

**멀티턴(multi-turn):** 한 대화 안에서 질문과 답이 여러 번 오가는 모양입니다. system 하나 뒤에 user·assistant 쌍을 이어 붙이며, 뒤 질문이 앞 답에 기대는 대화를 가르칠 때 씁니다.

- **user:** "연차는 며칠 생기나요?"
- **assistant:** "1년간 80퍼센트 이상 출근하면 15일의 유급휴가가 생깁니다."
- **user:** "입사 1년이 안 됐으면요?"
- **assistant:** "1개월 개근하면 1일의 유급휴가가 생깁니다."

둘째 질문은 앞 답(1년간 출근)을 이어받아야 뜻이 통합니다. 이 교안의 데이터셋은 한 번 묻고 답하는 행으로 둡니다.

### 6-2. 학습 텍스트와 추론 프롬프트

지난 시간에 `apply_chat_template`으로 messages를 모델이 아는 문자열로 바꿨습니다. 파인튜닝에서는 같은 함수로 문자열 두 개를 만듭니다.

- **학습 텍스트(training text):** system·user·assistant를 모두 넣은 문자열입니다. 학습 때 모델에 들어가며, `add_generation_prompt`는 기본값 `False`로 둡니다.
- **추론 프롬프트(inference prompt):** 마지막 assistant 메시지를 뺀 messages에 `add_generation_prompt=True`를 주어, 끝에 모델이 답할 자리를 연 문자열입니다. 학습한 모델에 질문할 때 들어갑니다.
- **chat template 정합:** 학습 텍스트가 추론 프롬프트로 정확히 시작하는 상태입니다. 앞부분이 다르면 모델은 추론 때 학습에서 본 적 없는 모양의 입력을 받습니다.

<img src="images/11_template_alignment.png" width="1050" alt="제목은 'chat template 정합: 학습 텍스트는 추론 프롬프트로 시작합니다'입니다. 위 띠 '추론 프롬프트 (add_generation_prompt=True)'에는 '<|im_start|>system 너는 노무 상담 챗봇이다. …<|im_end|>', '<|im_start|>user 질문<|im_end|>', '<|im_start|>assistant <think></think>' 세 칸이 있고, 세 번째 칸은 '<|im_start|>assistant' 아래 줄에 빈 think 블록 '<think></think>'가 있습니다. 아래 띠 '학습 텍스트 (add_generation_prompt=False, 기본값)'에는 빈 think 블록까지 같은 세 칸이 같은 자리에 있고, 뒤에 초록 칸 '답 문장<|im_end|>'가 하나 더 있습니다. 두 띠의 세 칸 아래에는 괄호와 '같은 앞부분'이 있고, 초록 칸 아래에는 '남는 부분: 답 + <|im_end|>'가 있습니다. 서류철을 든 노무 담당자가 계산 결과 상자 'startswith: True'를 가리킵니다.">

토크나이저는 다음 시간에 파인튜닝할 모델인 `Qwen/Qwen3.5-0.8B`의 것을 씁니다. 모델 가중치 없이 토크나이저 파일만 내려받으므로 GPU가 없어도 됩니다.

In [ ]:
from transformers import AutoTokenizer

# 다음 시간에 파인튜닝할 모델의 토크나이저. 처음 실행하면 토크나이저 파일만 내려받습니다
chat_tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen3.5-0.8B")

# 첫 행의 학습 텍스트(add_generation_prompt 기본값 False)
sample_messages = sample_chat["messages"]
training_text = chat_tokenizer.apply_chat_template(sample_messages, tokenize=False)  # tokenize=False: 토큰 id 대신 문자열로
print(training_text)

In [ ]:
# 추론 프롬프트: 마지막 assistant 메시지를 빼고 모델이 답할 자리를 엽니다
inference_prompt = chat_tokenizer.apply_chat_template(sample_messages[:-1], tokenize=False, add_generation_prompt=True)  # 답을 빼고 답할 자리를 엶
print(inference_prompt)

In [ ]:
# 학습 텍스트가 추론 프롬프트로 시작하는지, 그 뒤에 남는 부분은 무엇인지 확인합니다
answer_part = training_text[len(inference_prompt):]  # 추론 프롬프트 뒤에 남는 부분
print("앞부분 일치:", training_text.startswith(inference_prompt))
print("남는 부분:", repr(answer_part))
print("끝 토큰:", chat_tokenizer.eos_token)

**결과 읽기**
- **앞부분 일치 `True`:** 학습 텍스트가 추론 프롬프트로 시작합니다. 두 문자열 모두 답 앞에 빈 생각 블록 `<think>\n\n</think>\n\n`이 있어(이 토크나이저는 기본으로 생각을 끔), 모델은 빈 블록 바로 뒤에서 답을 시작하도록 배웁니다.
- **남는 부분과 끝 토큰:** 답 문장 뒤에 `eos_token`(end of sequence, 끝 표시)인 `<|im_end|>`와 줄바꿈이 붙어, 모델은 답을 끝내는 자리까지 배웁니다.
- **`add_generation_prompt`:** 학습 텍스트는 기본값 `False`로 만듭니다. `True`를 주면 답 뒤에 빈 assistant 차례가 더 붙어 모델이 그 모양까지 배웁니다([Hugging Face chat template 문서](https://huggingface.co/docs/transformers/main/en/chat_templating)).

### 6-3. 전체 행의 chat 포맷과 토큰 길이

첫 행에서 한 방법을 균형을 맞춘 행 전체에 적용합니다. 바꾼 행은 분할과 저장에 쓰고, 다음 시간에 학습 설정의 최대 길이를 정하려고 학습 텍스트의 토큰 수를 셉니다.

- **`chat_rows`:** 균형을 맞춘 행을 복사해 `messages`를 더한 행입니다. 분할에 쓸 `origin_id`와 저장할 키(`question`·`answer` 등)를 그대로 둡니다.
- **토큰 수:** `chat_tokenizer(text, add_special_tokens=False)["input_ids"]`의 길이입니다. 학습 텍스트에 특수 토큰이 이미 들어 있어 `add_special_tokens=False`로 둡니다.

In [ ]:
# 균형을 맞춘 행마다 원래 키를 그대로 두고 messages를 더합니다
chat_rows = []
for row in balanced:
    chat_row = dict(row)
    chat_row["messages"] = to_chat(row)["messages"]
    chat_rows.append(chat_row)

# 학습 때 실제로 들어가는 문자열(학습 텍스트)로 토큰 수를 셉니다
texts = [chat_tokenizer.apply_chat_template(row["messages"], tokenize=False) for row in chat_rows]

length_table = pd.DataFrame(chat_rows, columns=["id", "method", "question", "answer"])
length_table["토큰 수"] = [len(chat_tokenizer(text, add_special_tokens=False)["input_ids"]) for text in texts]  # 템플릿의 특수 토큰을 또 붙이지 않음

# 토큰 수 분포(개수, 평균, 표준편차, 최솟값, 사분위수, 최댓값)
token_summary = length_table["토큰 수"].describe()
display(token_summary.round(1))

**결과 읽기**
- **`max`와 `75%`:** `max`는 가장 긴 학습 텍스트의 토큰 수이고, `75%`는 학습 텍스트 4개 가운데 3개가 이 토큰 수 이하라는 뜻입니다. 다음 시간에 학습 설정의 최대 길이를 정할 때 봅니다.

## 7. 학습/검증 분할

학습 중 처음 보는 질문에 얼마나 잘 답하는지 확인하려고 일부 행을 검증 데이터로 떼어 둡니다. 증강으로 같은 원본에서 나온 행이 여러 개라, 나누는 방법에 따라 검증 데이터가 학습 데이터와 겹칩니다.

### 7-1. 행 단위 분할의 누수

**데이터 누수(data leakage):** PyTorch 신경망 단원에서 본 것처럼 검증 데이터의 정보가 학습 데이터에 섞이는 일입니다.

<img src="images/14_split_row_leak.gif" width="1050" alt="움직이는 그림입니다. 제목은 '학습/검증 분할: 행 단위로 나누면 생기는 누수'입니다. 첫 장면에서는 원본 A~F마다 '원본', '패러프레이즈', '역번역' 카드 세 장이 같은 색이고, 18장이 섞여 있습니다. 노란 띠에 'round(18 x 0.2) = 4행을 검증으로 보냅니다'가 적혀 있습니다. 둘째 장면에서는 '학습 14행' 판과 '검증 4행' 판(E 패러프레이즈, E 원본, B 패러프레이즈, D 원본)으로 나뉩니다. 셋째 장면에서는 오른쪽 위에 빨간 '누수 4행'이 생기고, 검증 카드 네 장에 빨간 '누수' 배지가, 같은 원본 글자의 학습 카드에 빨간 테두리가 붙습니다. 노란 띠에는 '누수 4행: 학습에서 본 문장의 변형으로 검증하니 점수가 부풀려집니다'가 적혀 있습니다.">

1. 원본 질문 "연차는 며칠인가요?"와 그 패러프레이즈 "연차휴가는 며칠 받을 수 있나요?"가 같은 `origin_id`로 있습니다.
2. 행 단위로 섞어 나누면 원본은 검증, 패러프레이즈는 학습에 갈 수 있습니다.
3. 모델은 검증 질문의 다른 표현과 그 답을 이미 학습했으므로, 검증 답을 맞혀도 처음 보는 질문을 맞힌 것이 아닙니다.

행 단위로 섞어 앞 20%를 검증 데이터로 나누고, 검증 행 가운데 같은 `origin_id`가 학습 데이터에도 있는 행을 셉니다.

In [ ]:
# 검증 비율. 원본 단위 분할, 따라하기, 데이터 카드에 같은 값을 씁니다(난수 시드 SPLIT_SEED는 주제 상한을 적용할 때 정했습니다)
VAL_RATIO = 0.2


def count_leaked(train, val):
    """검증 행 가운데 같은 origin_id가 학습 데이터에도 있는 행 수를 돌려줍니다."""
    train_origins = {row["origin_id"] for row in train}
    leaked = 0
    for row in val:
        if row["origin_id"] in train_origins:
            leaked += 1
    return leaked


# 행 단위로 섞어 앞 20%를 검증 데이터로 씁니다
shuffled_rows = list(chat_rows)
random.Random(SPLIT_SEED).shuffle(shuffled_rows)
row_val_count = round(len(shuffled_rows) * VAL_RATIO)  # 검증 행 수 = 전체 행의 20%
row_val = shuffled_rows[:row_val_count]
row_train = shuffled_rows[row_val_count:]

print("행 단위 분할 누수(학습에 같은 원본이 있는 검증 행 수):", count_leaked(row_train, row_val))

증강 행이 있으면 이 수는 0이 아닐 수 있고, 값은 증강 결과에 따라 달라집니다.

### 7-2. 원본 단위 분할

**원본 단위 분할:** PyTorch 신경망 단원의 묶음 단위 분할(group split)에서 묶음을 원본 Q&A(같은 `origin_id`)로 정한 분할입니다. 원본과 거기서 늘린 행을 한 묶음으로 보고, 묶음째 학습이나 검증 한쪽에 넣습니다.

- **`split_by_group(rows, key, val_ratio, seed)`:** `key` 값이 같은 행을 한 묶음으로 봅니다. 묶음 이름을 정렬해 `random.Random(seed)`로 섞은 뒤, 앞 `round(묶음 수 * val_ratio)`개 묶음을 검증으로 보내 `(train, val)`을 돌려줍니다.
- 정렬해 두면 행 순서와 상관없이 같은 시드에서 같은 묶음이 뽑힙니다.
- `val_ratio`는 행이 아니라 묶음 수에 곱하므로, 검증 행의 비율은 묶음 크기에 따라 0.2에서 조금 벗어납니다.

<img src="images/14_split_group.gif" width="1050" alt="움직이는 그림입니다. 제목은 '학습/검증 분할: 원본 묶음 단위로 나누기'입니다. 첫 장면에서는 A~F 여섯 묶음이 각각 '원본', '패러프레이즈', '역번역' 카드 세 장씩 한 상자에 들어 있고, 노란 띠에 '원본 단위로 묶습니다: 같은 색 3장이 항상 함께 움직입니다'가 적혀 있습니다. 둘째 장면에서는 '학습 15행 (5묶음)' 판(A, B, C, E, F)과 '검증 3행' 판(D 묶음)으로 나뉘고, 오른쪽 위에 초록 '누수 0행'이 나타납니다. 노란 띠에는 'round(6 x 0.2) = 1묶음을 검증으로 보냅니다'와 '학습 15행, 검증 3행: 같은 색이 양쪽에 걸치지 않아 누수 0행'이 적혀 있습니다.">

In [ ]:
def split_by_group(rows, key, val_ratio, seed):
    """key 값이 같은 행을 한 묶음으로 섞어 나눈 (학습 행 리스트, 검증 행 리스트)를 돌려줍니다."""
    # 묶음 이름을 정렬한 뒤 난수 시드로 섞습니다
    groups = sorted({row[key] for row in rows})  # set은 순서가 없어 정렬한 뒤 섞음
    random.Random(seed).shuffle(groups)

    # 앞쪽 묶음을 검증 묶음으로 씁니다
    val_groups = set(groups[:round(len(groups) * val_ratio)])  # 묶음 수의 val_ratio만큼이 검증 묶음

    train = []
    val = []
    for row in rows:
        if row[key] in val_groups:
            val.append(row)
        else:
            train.append(row)
    return train, val


# 원본 Q&A 묶음의 20%를 검증 데이터로 나눕니다. 오염 검사를 마친 행이라 이 둘이 최종 학습, 검증 데이터입니다
train_rows, val_rows = split_by_group(chat_rows, key="origin_id", val_ratio=VAL_RATIO, seed=SPLIT_SEED)
print("학습 행 수:", len(train_rows), "| 검증 행 수:", len(val_rows))
print("원본 단위 분할 누수:", count_leaked(train_rows, val_rows))

원본 단위 분할은 같은 원본의 행이 늘 한쪽에만 있으므로, 같은 `origin_id` 기준의 누수(`count_leaked`)는 항상 0입니다. 같은 조문(`doc_id`)에서 나온 다른 원본 Q&A는 학습과 검증에 갈릴 수 있어, 검증 행 가운데 일부는 학습에 있는 조문을 다른 질문으로 묻는 행입니다.

`key="source"`로 부르면 검증 데이터에 학습에서 한 행도 보지 못한 주제만 들어가, 처음 보는 주제에 대한 일반화를 확인하는 더 엄격한 분할이 됩니다(이 교안은 `origin_id` 분할을 씁니다).

## 8. 저장과 데이터 카드

최종 학습·검증 데이터를 학습 단계에서 바로 읽을 수 있게 파일로 저장하고, 데이터셋을 넘겨받는 사람이 출처와 처리 과정을 알 수 있게 기록을 남깁니다.

### 8-1. JSONL로 저장하고 다시 읽기

학습·검증 두 파일을 같은 방법으로 쓰므로 `save_jsonl` 함수로 둡니다. 행의 키(`id`, `origin_id`, `doc_id`, `source`, `method`, `question`, `answer`, `messages`)를 그대로 저장합니다. 기존 FAQ 행에는 `doc_id`가 없고, `question_type`은 문서 생성 행과 그 증강 행에만 있습니다.

In [ ]:
def save_jsonl(rows, path):
    """행 딕셔너리 리스트를 한 줄에 JSON 하나씩 파일로 저장합니다."""
    with open(path, "w", encoding="utf-8") as file:
        for row in rows:
            # ensure_ascii=False 로 한글을 \uXXXX 대신 글자 그대로 씁니다
            file.write(f"{json.dumps(row, ensure_ascii=False)}\n")


# 최종 학습, 검증 데이터를 저장합니다
train_path = output_dir / "labor_train.jsonl"
save_jsonl(train_rows, train_path)
save_jsonl(val_rows, output_dir / "labor_val.jsonl")

# 저장한 학습 파일을 다시 읽어 첫 행을 봅니다
reloaded_train = load_jsonl(train_path)
print("다시 읽은 학습 행 수:", len(reloaded_train))
display(reloaded_train[0])

### 8-2. 데이터 카드

**데이터 카드(data card):** 데이터셋의 목적·출처·처리 과정·규모·한계를 적은 문서입니다. 데이터셋을 받은 사람이 어떻게 만들어졌는지 확인하고 같은 과정을 다시 할 수 있습니다.

- **근거:** 데이터셋마다 이런 문서를 붙이자는 제안은 Datasheets for Datasets([Gebru et al., 2018](https://arxiv.org/abs/1803.09010))이고, Data Cards([Pushkarna et al., 2022](https://arxiv.org/abs/2204.01075))가 담을 항목과 형식을 정리했습니다.
- **실무:** Hugging Face Hub는 데이터셋 저장소마다 [데이터셋 카드](https://huggingface.co/docs/hub/datasets-cards)를 둡니다.
- 숫자는 손으로 적지 않고 앞에서 만든 변수로 채워, 데이터가 바뀌어도 카드가 실제와 같게 둡니다.

먼저 출처 항목에 적을 근거 법령과 시행일자를 모읍니다.

In [ ]:
# 법령이 개정되면 데이터를 다시 만들어야 하므로, Q&A의 근거 법령과 시행일자를 카드에 적습니다
law_sources = json.loads((data_dir / "law_sources.json").read_text(encoding="utf-8"))
used_laws = {row["doc_id"].split(" ")[0] for row in qa_rows}  # doc_id "근로기준법 제60조 제1항"의 앞 낱말

# 범위 밖 행의 doc_id "없음"은 law_sources에 없는 이름이라 아래 반복에서 걸러집니다
law_lines = []
for short_name, law in law_sources.items():
    if short_name in used_laws:
        law_lines.append(f"  - {law['법령명']} (시행일자 {law['시행일자']})")
law_text = "\n".join(law_lines)
print(law_text)

최종 규모와 검증 항목에 적을 값을 계산합니다.

- **최종 규모:** 만든 방법(`method`)별 학습·검증 행 수, 균형 뒤 주제별 행 수, 범위 밖 행의 비율과 거절 설계를 적어, 한 방법이나 주제가 한쪽에만 몰리지 않았는지 보게 합니다.
- **검증:** 학습·검증 사이 같은 원본 행 수(`count_leaked`)를 적습니다.

In [ ]:
# 만든 방법별 최종 학습, 검증 행 수
final_frame = pd.DataFrame(train_rows + val_rows)
final_frame["분할"] = ["학습"] * len(train_rows) + ["검증"] * len(val_rows)
method_split_table = pd.crosstab(final_frame["method"], final_frame["분할"])  # 방법별 학습, 검증 행 수 테이블

method_lines = []
for method, counts in method_split_table.iterrows():
    method_lines.append(f"  - {method}: 학습 {counts['학습']}건, 검증 {counts['검증']}건")
method_text = "\n".join(method_lines)

# 최종 데이터에서 범위 밖 행의 비율(%). 비교 결과 True의 평균이 곧 그 비율입니다
refusal_percent = round((final_frame["method"] == "범위 밖").mean() * 100, 1)

# 학습 텍스트의 최대 토큰 수와 학습, 검증 사이 같은 원본 행 수
max_tokens = length_table["토큰 수"].max()
final_leaked = count_leaked(train_rows, val_rows)
print("범위 밖 비율(%):", refusal_percent, "| 최대 토큰 수:", max_tokens, "| 같은 원본 행 수:", final_leaked)

모은 값으로 마크다운 카드를 만들어 데이터셋 파일과 같은 폴더에 저장합니다.

- **처리 단계:** 증강, 품질 필터, 오염 검사, 구성 균형, 포맷, 분할을 이 교안에서 한 순서대로 적습니다.

In [ ]:
# 앞 단계의 실제 수치로 마크다운 데이터 카드를 만듭니다
dataset_card = f"""# 노무 상담 Q&A 파인튜닝 데이터셋

## 목적
- 노동법 조문에 근거해 존댓말로 간결히 답하는 노무 상담 챗봇의 지도 파인튜닝(SFT)

## 출처
- 국가법령정보센터(law.go.kr) 노동법 조문으로 교안 01에서 만든 Q&A {len(qa_rows)}건(문서 생성, 질문 늘리기, 범위 밖 거절), 생성용 LLM {model_name}
- 수업용으로 만든 기존 FAQ {len(faq_rows)}건
- 기준 법령:
{law_text}

## 처리 단계
1. 증강: 패러프레이즈 {len(paraphrase_rows)}건, 역번역 {len(back_translation_rows)}건(표본 200, 시드 {SAMPLE_SEED}) 가운데 숫자가 같은 {len(augmented_rows)}건을 더함
2. 품질 필터: {len(raw)}건 -> 형식 {len(after_format)}건 -> 길이(답 5글자 이상) {len(after_len)}건 -> 정확 중복 {len(after_dedup)}건 -> 유사 중복(글자 자카드 0.8) {len(clean)}건
3. 오염 검사: 평가 질문 {len(eval_questions)}개와 글자 3-gram 겹침 {CONTAMINATION_THRESHOLD} 이상인 원본 묶음 {len(contaminated_origins)}개를 뺌 {len(clean)}건 -> {len(decontaminated)}건
4. 구성 균형: 주제별 상한 {TOPIC_CAP}행(중앙값의 1.5배), 증강 행부터 빼고 그래도 넘으면 원본 묶음째 줄임 {len(decontaminated)}건 -> {len(balanced)}건
5. 포맷: chat 포맷(messages), system 문장 "{SYSTEM_PROMPT}", Qwen/Qwen3.5-0.8B 학습 텍스트 최대 {max_tokens}토큰
6. 분할: 원본 Q&A(origin_id) 단위, 검증 묶음 비율 {VAL_RATIO}, 난수 시드 {SPLIT_SEED}

## 최종 규모
- 학습 데이터 {len(train_rows)}건, 검증 데이터 {len(val_rows)}건
- 균형 뒤 주제별 행 수(범위 밖 제외): 최소 {balanced_topic_counts.min()}행, 최대 {balanced_topic_counts.max()}행
- 범위 밖(거절) 행 비율: {refusal_percent}%
- 거절 설계: 고정 거절 답 하나, 범위 밖 주제 다섯 가지(연말정산 같은 세금, 실업급여와 고용보험, 4대보험 가입, 이직 준비, 개인 투자)로 만든 질문
- 만든 방법별:
{method_text}

## 검증
- 학습·검증 사이 같은 원본(origin_id): {final_leaked}행

## 한계
- LLM이 만든 질문과 답이 섞여 있어, 숫자 대조로 잡지 못한 사실 오류가 남아 있을 수 있습니다.
- 조문 하나 단위로 Q&A를 만들어, 다른 조문의 조건(예: 상시 4명 이하 사업장 적용 제외)이 빠진 답이 있을 수 있습니다.
- 질문 늘리기 행은 검색이 근거 조문을 놓치면 빠지므로, 검색이 약한 주제의 새 질문이 적게 남았을 수 있습니다.
- 질문 유형(question_type)은 생성용 LLM이 붙인 라벨이라 틀린 라벨이 섞여 있을 수 있습니다.
- 유사 중복은 글자 집합으로만 걸러, 뜻이 같은 다른 표현은 남아 있고 낱말 하나 다른 질문이 함께 빠졌을 수 있습니다.
- 글자 3-gram 오염 검사는 표현이 다른 같은 질문을 잡지 못합니다.
- 기존 FAQ 답은 조문과의 숫자 대조를 거치지 않았습니다.
- 법령이 개정되면 바뀐 조문으로 데이터를 다시 만들어야 합니다.
"""

# 데이터셋 파일과 같은 폴더에 저장해 함께 넘깁니다
(output_dir / "labor_dataset_card.md").write_text(dataset_card, encoding="utf-8")
print(dataset_card)

## 9. 🖐️ 함께 따라하기: 전월세 상담 데이터셋 다듬기

본실습의 증강, 필터 네 단계, 오염 검사, 구성 점검과 주제 상한, `to_chat`, `split_by_group`을 본실습과 같은 순서로 전월세 상담 Q&A에 적용합니다. 같은 다듬기 과정이 다른 분야의 데이터에서도 그대로 도는지 직접 작성해 확인합니다.

**본실습과 다른 점**
- **데이터:** 주택임대차보호법 조문(국가법령정보센터 law.go.kr)으로 수업용으로 만든 전월세 상담 Q&A 192행(`data/따라하기/lease_qa.jsonl`)입니다. 문서 생성·패러프레이즈·역번역·기존 FAQ 행이 이미 합쳐져 있고 `origin_id`가 붙어 있습니다. 범위 밖 행, `doc_id`, `question_type`은 없습니다.
- **평가 질문:** 20개(`data/따라하기/lease_eval_questions.json`)입니다.
- **system 문장:** 전월세 상담 챗봇 문장(`SYSTEM_PROMPT_LEASE`)입니다.
- **증강은 표본으로 연습:** 원본 10행으로 패러프레이즈와 역번역을 만들어 보며, 이때만 생성용 LLM을 부릅니다(최대 30회). 그 뒤 단계는 증강 행이 이미 들어 있는 파일로 이어 가므로 누가 실행해도 결과가 같습니다.

**실습 진행 방법:** 작성 셀의 `...`만 주석에 맞춰 채웁니다. 증강, 품질 필터 적용, 오염 검사, 주제 상한, chat 포맷과 학습 텍스트·추론 프롬프트, 원본 단위 분할은 본실습에서 한 방법을 그대로 씁니다. 데이터 읽기와 JSONL 저장 셀은 완성 코드라 실행만 하고, 완성 코드는 `정답/교안_02_정답.ipynb`의 같은 절에 있습니다.

**설정**
- 증강 연습 표본 `LEASE_AUGMENT_IDS`(문서 생성 행 a022~a031), 동시 요청 상한 `MAX_CONCURRENCY`(32)
- 길이 필터 `min_a=5`, 유사 중복 기준 0.8(`dedup_similar`)
- 오염 기준 `CONTAMINATION_THRESHOLD`(0.7)
- 주제 상한: 주제별 행 수 중앙값의 1.5배를 `round`로 정수로 만든 값(.5는 짝수 쪽), `cap_by_topic`, 시드 `SPLIT_SEED`(42)
- 분할 `key="origin_id"`, `VAL_RATIO`(0.2), `SPLIT_SEED`(42)

### 9-1. 데이터 읽기

In [ ]:
# 전월세 상담 Q&A 192행과 평가 질문 20개
lease_rows = load_jsonl(data_dir / "따라하기" / "lease_qa.jsonl")
lease_eval_questions = json.loads((data_dir / "따라하기" / "lease_eval_questions.json").read_text(encoding="utf-8"))

# 전월세 상담 챗봇의 system 문장. 학습과 추론에 같은 문장을 씁니다
SYSTEM_PROMPT_LEASE = "너는 전월세 상담 챗봇이다. 주택임대차보호법 조문에 근거해 존댓말로 간결히 답한다."

lease_table = pd.DataFrame(lease_rows)
print("행 수:", len(lease_rows), "| 평가 질문 수:", len(lease_eval_questions))
display(lease_table.head())

### 9-2. 증강 연습

본실습과 같은 방법으로 질문 표현을 늘려 봅니다. `lease_rows`에는 패러프레이즈와 역번역 행이 이미 들어 있으므로, 여기서는 아직 증강하지 않은 것처럼 문서 생성 행 10개(`LEASE_AUGMENT_IDS`)만 표본으로 씁니다.

- **패러프레이즈:** 본실습의 `Paraphrase` 스키마를 쓰고, 프롬프트는 말투 규칙만 세입자 말투로 바꿉니다. 질문마다 하나씩 요청합니다(모델 호출 10회).
- **역번역:** 본실습의 `translator`로 영어 번역 뒤 한국어 번역을 합니다(모델 호출 최대 20회).
- **연습 결과는 따로 둡니다:** 여기서 만든 행은 뒤 단계에 넣지 않습니다. 품질 필터부터는 증강 행이 이미 들어 있는 `lease_rows`로 이어 가므로, 뒤 확인 기준의 숫자는 이 연습 결과와 관계없이 같습니다.

In [ ]:
# 🖐️ 함께 따라하기: 원본 10행의 질문마다 패러프레이즈를 하나씩 만드세요

# 증강 연습 표본: 문서 생성 행 a022~a031 10행(숫자가 든 질문 a024, a025 포함)
LEASE_AUGMENT_IDS = ["a022", "a023", "a024", "a025", "a026", "a027", "a028", "a029", "a030", "a031"]
lease_augment_sample = []
for row in lease_rows:
    if row["id"] in LEASE_AUGMENT_IDS:
        lease_augment_sample.append(row)

# 원본 질문을 id로 찾는 딕셔너리(증강 질문과 나란히 볼 때와 숫자 확인에서 씁니다)
lease_question_by_id = {row["id"]: row["question"] for row in lease_augment_sample}

# 본실습 프롬프트에서 말투 규칙만 세입자 말투로 바꿉니다
lease_paraphrase_prompt = ChatPromptTemplate.from_messages([
    ("system", "전월세 상담 챗봇 학습 데이터의 질문을 뜻은 그대로 두고 다른 표현으로 바꿔 써야 한다. 바꿔 쓴 질문에는 원래 질문의 답이 그대로 붙는다."),
    ("human", """아래 [문장]을 뜻은 그대로 두고 표현(어순, 낱말, 말투)만 바꿔 한 문장으로 다시 써라.

뜻 지키기:
1. 숫자, 기간, 이름은 원문 그대로 둔다.
2. 새로운 정보를 더하거나 빼지 않는다.

말투:
3. 세입자가 실제로 물어볼 말투로 쓴다.

[문장]
{text}"""),
])

# 1) lease_paraphrase_prompt와 Paraphrase 모양으로만 답하는 llm을 이은 체인 쓰기
lease_paraphrase_generator = ...

# 2) lease_paraphrase_inputs를 .batch로 보내기: config에 max_concurrency(MAX_CONCURRENCY), return_exceptions=True (모델 호출 10회)
lease_paraphrase_inputs = [{"text": row["question"]} for row in lease_augment_sample]
lease_paraphrase_results = ...

lease_paraphrase_rows = []
lease_paraphrase_skipped = 0
for row, result in zip(lease_augment_sample, lease_paraphrase_results):
    # 실패한 요청은 결과 자리에 예외 객체가 있으므로 건너뜁니다
    if isinstance(result, Exception):
        lease_paraphrase_skipped += 1
        continue

    # 원본 행을 복사해 바뀌는 키만 고칩니다
    new_row = dict(row)
    new_row["id"] = f"{row['id']}-p"  # 원본 id 뒤에 -p, origin_id는 원본 그대로
    new_row["method"] = "패러프레이즈"
    new_row["question"] = result.text
    lease_paraphrase_rows.append(new_row)

print("표본 행 수:", len(lease_augment_sample), "| 결과 수:", len(lease_paraphrase_results))
print("패러프레이즈 행 수:", len(lease_paraphrase_rows), "| 건너뛴 요청 수:", lease_paraphrase_skipped)
# 새 질문(question) 옆에 원본 질문을 붙여 봅니다
lease_paraphrase_table = pd.DataFrame(lease_paraphrase_rows)
lease_paraphrase_table["원본 질문"] = [lease_question_by_id[row["origin_id"]] for row in lease_paraphrase_rows]
display(lease_paraphrase_table)

In [ ]:
# 🖐️ 함께 따라하기: 같은 표본 10행을 영어로 번역했다가 한국어로 다시 번역해 역번역 행을 만드세요

# 1) lease_english_inputs를 본실습의 translator로 .batch에 보내기: config에 max_concurrency(MAX_CONCURRENCY), return_exceptions=True (모델 호출 10회)
lease_english_inputs = [{"language": "영어", "text": row["question"]} for row in lease_augment_sample]
lease_english_results = ...

# 영어 번역에 성공한 행과 영어 문장만 같은 순서로 모읍니다
lease_english_rows = []
lease_english_texts = []
for row, result in zip(lease_augment_sample, lease_english_results):
    if isinstance(result, Exception):
        continue
    lease_english_rows.append(row)
    lease_english_texts.append(result.text)

# 2) lease_korean_inputs를 translator로 .batch에 보내기: 1)과 같은 config와 return_exceptions=True (모델 호출: 영어 번역에 성공한 수만큼)
lease_korean_inputs = [{"language": "한국어", "text": text} for text in lease_english_texts]
lease_korean_results = ...

# 패러프레이즈와 같은 방법으로 역번역 행을 만듭니다
lease_back_translation_rows = []
for row, result in zip(lease_english_rows, lease_korean_results):
    if isinstance(result, Exception):
        continue
    new_row = dict(row)
    new_row["id"] = f"{row['id']}-b"  # 원본 id 뒤에 -b
    new_row["method"] = "역번역"
    new_row["question"] = result.text
    lease_back_translation_rows.append(new_row)

print("영어 번역 성공 수:", len(lease_english_rows), "| 역번역 행 수:", len(lease_back_translation_rows))

# 역번역 질문(question) 옆에 원본 질문과 중간의 영어 문장을 붙여 봅니다
lease_english_by_id = {row["id"]: text for row, text in zip(lease_english_rows, lease_english_texts)}
lease_back_translation_table = pd.DataFrame(lease_back_translation_rows)
lease_back_translation_table["원본 질문"] = [lease_question_by_id[row["origin_id"]] for row in lease_back_translation_rows]
lease_back_translation_table["영어"] = [lease_english_by_id[row["origin_id"]] for row in lease_back_translation_rows]
display(lease_back_translation_table)

증강한 질문이 원본과 같은 숫자를 묻는지 확인합니다. 증강 행은 원본 행을 `dict(row)`로 복사하고 질문만 바꿨으므로 답과 `origin_id`는 원본 그대로입니다.

In [ ]:
# 🖐️ 함께 따라하기: 증강 행이 원본과 같은 숫자를 묻는지 확인하세요

# 원본 질문은 패러프레이즈 셀에서 만든 lease_question_by_id로 찾습니다
lease_augmented_rows = []
lease_augment_dropped = []
for row in lease_paraphrase_rows + lease_back_translation_rows:
    original_question = lease_question_by_id[row["origin_id"]]

    # 원본 질문과 증강 질문(row["question"])의 숫자 리스트를 각각 정렬해 같은지 비교하는 식 쓰기
    same_numbers = ...

    if same_numbers:
        lease_augmented_rows.append(row)
    else:
        lease_augment_dropped.append({"id": row["id"], "원본 질문": original_question, "증강 질문": row["question"]})

print("증강 행 수:", len(lease_paraphrase_rows + lease_back_translation_rows))
print("통과한 증강 행 수:", len(lease_augmented_rows), "| 숫자가 달라 걸린 행 수:", len(lease_augment_dropped))
display(pd.DataFrame(lease_augment_dropped, columns=["id", "원본 질문", "증강 질문"]))

**확인 기준**
- **패러프레이즈:** 표본 행 수와 결과 수가 10이고, 패러프레이즈 행 수와 건너뛴 요청 수의 합이 10입니다.
- **역번역:** 역번역 행 수가 영어 번역 성공 수 이하입니다(실패가 없으면 둘 다 10).
- **통과와 걸림:** 통과한 증강 행 수와 숫자가 달라 걸린 행 수의 합이 증강 행 수와 같습니다.
- **id:** 새 행의 `id`는 원본 id 뒤에 `-p`나 `-b`, `origin_id`는 원본 id 그대로입니다.
- **걸린 행:** 숫자가 실제로 바뀌었는지, "두 번"을 "2번"으로 쓴 것처럼 적는 방식만 바뀌었는지 읽어 판정합니다(걸린 수는 실행마다 다르고 0일 수도 있음).

### 9-3. 품질 필터 네 단계

In [ ]:
# 🖐️ 함께 따라하기: 품질 필터 네 단계를 차례로 적용하세요

# 1) 형식 검증을 통과한 행만 담기
lease_after_format = []
for row in lease_rows:
    # format_ok(row)가 True이면 row를 lease_after_format에 append하기
    ...

# 2) 길이 필터를 통과한 행만 담기
lease_after_len = []
for row in lease_after_format:
    # length_ok(row)가 True이면 row를 lease_after_len에 append하기
    ...

# 3) 정확 중복 제거: dedup_exact에 lease_after_len을 넣기
lease_after_dedup = ...

# 4) 유사 중복 제거: dedup_similar에 lease_after_dedup을 넣기
lease_clean = ...

print("단계별 남은 수:", len(lease_rows), "->", len(lease_after_format), "->", len(lease_after_len), "->", len(lease_after_dedup), "->", len(lease_clean))

In [ ]:
# 단계마다 버린 행의 id
lease_stages = [
    ("형식 검증", lease_rows, lease_after_format),
    ("길이 필터", lease_after_format, lease_after_len),
    ("정확 중복 제거", lease_after_len, lease_after_dedup),
    ("유사 중복 제거", lease_after_dedup, lease_clean),
]
for stage_name, before, after in lease_stages:
    stage_ids = [row["id"] for row in dropped_rows(before, after)]
    print(f"{stage_name}:", stage_ids)

### 9-4. 오염 검사

In [ ]:
# 🖐️ 함께 따라하기: 평가 질문과 겹치는 행을 찾아 그 원본 묶음을 빼세요

# 1) 평가 질문 20개와의 겹침 가운데 가장 큰 값이 기준 이상인 행 모으기
lease_contaminated = []
for row in lease_clean:
    best_overlap = 0
    for eval_question in lease_eval_questions:
        # overlap에 ngram_overlap(row["question"], eval_question)을 담고, overlap이 best_overlap보다 크면 best_overlap을 overlap으로 바꾸기
        ...
    if best_overlap >= CONTAMINATION_THRESHOLD:  # 가장 많이 겹친 평가 질문과 비교
        lease_contaminated.append({"id": row["id"], "origin_id": row["origin_id"], "question": row["question"], "겹침": best_overlap})

# 2) 걸린 원본 묶음을 뺀 행만 담기
lease_contaminated_origins = {row["origin_id"] for row in lease_contaminated}
lease_decontaminated = []
for row in lease_clean:
    # row["origin_id"]가 lease_contaminated_origins에 없으면 row를 lease_decontaminated에 append하기
    ...

print("오염 검사 전:", len(lease_clean), "-> 후:", len(lease_decontaminated))
lease_contaminated_table = pd.DataFrame(lease_contaminated, columns=["id", "origin_id", "question", "겹침"])
display(lease_contaminated_table.round(2))  # 겹침은 소수 둘째 자리까지

### 9-5. 구성 점검과 주제 상한

In [ ]:
# 🖐️ 함께 따라하기: lease_decontaminated의 구성을 점검하고, 본실습과 같은 상한 규칙으로 균형을 맞추세요
lease_decontaminated_table = pd.DataFrame(lease_decontaminated)

# 1) 방법별 행 수
lease_method_counts = lease_decontaminated_table["method"].value_counts()
display(lease_method_counts)

# 2) 주제별 행 수(범위 밖 제외)
lease_topic_counts = lease_decontaminated_table["source"].value_counts().drop(index="범위 밖", errors="ignore")  # 범위 밖이 없어도 오류 없이
display(lease_topic_counts)

# 3) 묶음 크기별 묶음 수
lease_bundle_sizes = lease_decontaminated_table["origin_id"].value_counts()
display(lease_bundle_sizes.value_counts().sort_index().rename_axis("묶음 크기").rename("묶음 수"))

# 4) 주제 상한: 주제별 행 수 중앙값의 1.5배
LEASE_TOPIC_CAP = round(lease_topic_counts.median() * 1.5)  # .5는 짝수 쪽으로 반올림

# 5) 증강 행부터, 그다음 원본 묶음째 줄이기: cap_by_topic에 lease_decontaminated, LEASE_TOPIC_CAP, SPLIT_SEED를 넣기
lease_balanced = ...
print("주제 상한(행):", LEASE_TOPIC_CAP, "| 균형 전:", len(lease_decontaminated), "-> 균형 후:", len(lease_balanced))

### 9-6. chat 포맷과 정합, 원본 단위 분할

In [ ]:
# 🖐️ 함께 따라하기: lease_balanced를 전월세 system 문장의 chat 행으로 바꿔 첫 행의 정합을 확인하고, 원본 단위로 나누세요

lease_chat_rows = []
for row in lease_balanced:
    chat_row = dict(row)

    # 1) 전월세 system 문장으로 messages 만들기: to_chat(row, system_prompt=SYSTEM_PROMPT_LEASE)의 "messages" 값을 담기
    chat_row["messages"] = ...
    lease_chat_rows.append(chat_row)

# 2) 첫 행의 학습 텍스트와 추론 프롬프트 만들기(apply_chat_template에 넣을 메시지를 먼저 봅니다)
lease_messages = lease_chat_rows[0]["messages"]
print("첫 chat 행:", lease_chat_rows[0]["id"])
display(pd.DataFrame(lease_messages))

# 학습 텍스트: chat_tokenizer.apply_chat_template에 lease_messages와 tokenize=False를 넣기
lease_training_text = ...

# 추론 프롬프트: chat_tokenizer.apply_chat_template에 lease_messages[:-1], tokenize=False, add_generation_prompt=True를 넣기
lease_inference_prompt = ...

print("앞부분 일치:", lease_training_text.startswith(lease_inference_prompt))
print("남는 부분:", repr(lease_training_text[len(lease_inference_prompt):]))  # 추론 프롬프트 뒤: 답 + 끝 토큰

# 3) 원본 단위 분할: split_by_group에 lease_chat_rows, key="origin_id", val_ratio=VAL_RATIO, seed=SPLIT_SEED를 넣기
lease_train, lease_val = ...
print("학습 행 수:", len(lease_train), "| 검증 행 수:", len(lease_val), "| 누수:", count_leaked(lease_train, lease_val))

**확인 기준**
- **품질 필터:** `단계별 남은 수: 192 -> 189 -> 186 -> 182 -> 178`이 출력됩니다.
- **버린 행:** 형식 검증 `a015`, `a025-p`(빈 질문)와 `a058-p`(`answer` 키 없음), 길이 필터 `a021`·`a034-p`·`a035-b`("가능.", "아니요.", "불가."), 정확 중복 `x02`·`x04`·`x05`·`x06`, 유사 중복 `x01`·`x03`·`x08`·`x09`입니다.
- **오염 검사:** `오염 검사 전: 178 -> 후: 174`가 출력되고 `x07`(겹침 1.0)·`x10`(0.86)·`x11`(0.83)·`x12`(0.86)가 걸립니다. 네 행 모두 평가 질문을 옮긴 기존 FAQ 행이라 원본 묶음이 자기 한 행입니다.
- **구성:** 방법별로 문서 생성 58, 패러프레이즈 47, 역번역 39, 기존 FAQ 30행, 주제별 30~38행이고, 묶음 크기는 1이 39개, 2가 18개, 3이 33개입니다.
- **주제 상한:** `주제 상한(행): 56 | 균형 전: 174 -> 균형 후: 174`가 출력됩니다(상한에 걸린 주제 없음).
- **정합:** 첫 chat 행(`a001`)의 앞부분 일치가 `True`이고, 남는 부분이 `'주거용 건물의 일부가 주거 외의 목적으로 사용되는 경우에도 주택임대차보호법이 적용됩니다.<|im_end|>\n'`입니다.
- **분할:** `학습 행 수: 137 | 검증 행 수: 37 | 누수: 0`이 출력됩니다.

확인 기준을 모두 맞췄으면 최종 학습·검증 데이터를 저장합니다. 다음 시간 따라하기가 이 두 파일을 씁니다.

In [ ]:
# 전월세 최종 학습, 검증 데이터를 본실습의 save_jsonl로 저장합니다(messages에 전월세 system 문장이 들어 있습니다)
save_jsonl(lease_train, output_dir / "lease_train.jsonl")
save_jsonl(lease_val, output_dir / "lease_val.jsonl")

print("lease_train.jsonl 행 수:", len(lease_train), "| lease_val.jsonl 행 수:", len(lease_val))
print("첫 행 system 문장:", lease_train[0]["messages"][0]["content"])

## 10. 핵심 정리

| 단계 | 하는 일 | 핵심 코드·기준 |
|---|---|---|
| 증강 | 질문 표현만 바꾸고 답은 원본 그대로, 숫자가 같은 행만 남김 | `paraphrase_generator`, `translator`(역번역은 표본 200개) |
| 품질 필터 | 형식 검증, 길이, 정확 중복, 유사 중복 순서로 거름 | `format_ok`, `length_ok`(답 5글자 이상), `dedup_exact`, `dedup_similar`(자카드 0.8) |
| 오염 검사 | 평가 질문과 겹치는 원본 묶음을 뺌 | `ngram_overlap`, `CONTAMINATION_THRESHOLD`(0.7) |
| 구성 점검과 균형 | 방법별, 주제별 행 수를 보고 주제 상한으로 줄임 | `cap_by_topic`, `TOPIC_CAP`(중앙값의 1.5배) |
| chat 포맷 | 균형을 맞춘 행을 system, user, assistant 메시지로 바꿈 | `to_chat`, `SYSTEM_PROMPT` |
| chat template 정합 | 학습 텍스트가 추론 프롬프트로 시작하는지 확인 | `apply_chat_template`, 추론 프롬프트는 `add_generation_prompt=True` |
| 토큰 길이 | 학습 텍스트의 토큰 수 분포를 봄 | `add_special_tokens=False`, `max`, `75%` |
| 분할 | 원본 Q&A 단위로 학습과 검증을 나눔 | `split_by_group`, `VAL_RATIO`(0.2), `count_leaked` 0 |
| 저장 | 학습, 검증 파일과 데이터 카드 | `labor_train.jsonl`, `labor_val.jsonl`, `labor_dataset_card.md` |

**꼭 기억할 것**
- 증강은 질문만 바꾸고 답은 원본 그대로 두며, 숫자가 바뀐 질문은 버립니다.
- 글자 기반 중복 제거와 오염 검사는 뜻을 보지 않으므로, 버린 행과 표현만 다른 같은 질문은 사람이 읽어 확인합니다.
- 학습 텍스트는 추론 때와 같은 system 문장과 같은 chat template으로 만들어야 학습한 대로 답합니다.
- 같은 원본에서 나온 행은 학습과 검증 가운데 한쪽에만 넣어야 검증 점수가 부풀지 않습니다.

다음 시간에는 `messages` 열을 가진 학습·검증 JSONL(질문·답·출처 키도 함께 남아 있습니다)로 TRL의 SFTTrainer 학습을 합니다.